In [ ]:
!pip install torch torchvision torchaudio
!pip install scikit-learn matplotlib pandas pillow tqdm
!pip install umap-learn
!pip install opencv-python


In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio


In [ ]:
import sys

!{sys.executable} -m pip install tqdm scikit-learn matplotlib pandas pillow umap-learn opencv-python


In [ ]:
# ===============================
# NESTED CROSS-VALIDATION PIPELINE
# MOBILENET V3 - BALANCED RESULTS
# FOR DATA IN BRIEF PUBLICATION
# ===============================

import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import time
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, 
                            classification_report, precision_score, recall_score,
                            balanced_accuracy_score)

# ===============================
# SETTINGS
# ===============================

DATA_DIR = "/path/to/Fish Dataset Raw"
OUTPUT_DIR = "./fish_outputs_mobilenet"
OUTER_FOLDS = 5
INNER_FOLDS = 2
RANDOM_SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32

# Set random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

print("="*70)
print("NESTED CROSS-VALIDATION WITH MOBILENET V3")
print("Bangladesh Fish Dataset - Data in Brief Publication")
print("="*70)
print(f"Device: {device}")
print(f"Outer Folds: {OUTER_FOLDS}")
print(f"Inner Folds: {INNER_FOLDS}")
print("="*70)


# ===============================
# MODEL DEFINITION
# ===============================

class MobileNetV3Classifier(nn.Module):
    """
    MobileNetV3-Small with custom classifier
    ~2.5M parameters - Good balance between size and performance
    """
    def __init__(self, num_classes, dropout=0.3, pretrained=True):
        super().__init__()
        
        if pretrained:
            self.backbone = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        else:
            self.backbone = mobilenet_v3_small(weights=None)
        
        # Freeze early layers for better generalization (optional - can reduce overfitting)
        # Uncomment below to freeze first few layers:
        # for param in list(self.backbone.parameters())[:-20]:
        #     param.requires_grad = False
        
        in_features = self.backbone.classifier[0].in_features
        
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.Hardswish(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)


def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Test model
test_model = MobileNetV3Classifier(num_classes=23, pretrained=True)
n_params = count_parameters(test_model)
print(f"\nModel: MobileNetV3-Small")
print(f"Trainable Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
del test_model


# ===============================
# HYPERPARAMETER SEARCH SPACE
# ===============================

HYPERPARAMETER_GRID = [
    {
        'name': 'Config_1',
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'epochs': 4,
        'dropout': 0.3,
        'label_smoothing': 0.1,
    },
    {
        'name': 'Config_2',
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'epochs': 4,
        'dropout': 0.4,
        'label_smoothing': 0.05,
    },
    {
        'name': 'Config_3',
        'lr': 5e-4,
        'weight_decay': 5e-5,
        'epochs': 3,
        'dropout': 0.35,
        'label_smoothing': 0.1,
    },
]

print("\nHyperparameter Configurations:")
for hp in HYPERPARAMETER_GRID:
    print(f"  {hp['name']}: lr={hp['lr']}, wd={hp['weight_decay']}, "
          f"epochs={hp['epochs']}, dropout={hp['dropout']}")


# ===============================
# HELPER FUNCTIONS
# ===============================

def remove_bangla_from_name(name):
    """Remove Bangla/Bengali characters from class names."""
    cleaned = re.sub(r'[\u0980-\u09FF]+', '', name)
    cleaned = re.sub(r'[^\x00-\x7F]+', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = cleaned.strip(' _-')
    cleaned = re.sub(r'\(\s*\)', '', cleaned)
    cleaned = cleaned.strip()
    return cleaned if cleaned else f"Species_{hash(name) % 1000}"


# ===============================
# DATASET
# ===============================

class FishDataset(Dataset):
    def __init__(self, root, transform=None, clean_names=True):
        self.root = Path(root)
        self.transform = transform
        
        classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.original_classes = classes.copy()
        
        if clean_names:
            self.cleaned_classes = [remove_bangla_from_name(c) for c in classes]
        else:
            self.cleaned_classes = classes.copy()
        
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: self.cleaned_classes[i] for i in range(len(classes))}
        self.idx_to_original_class = {i: classes[i] for i in range(len(classes))}
        
        self.samples = []
        self.class_counts = {}
        
        for c in classes:
            class_samples = []
            for img_path in (self.root / c).glob("*"):
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
                    class_samples.append((str(img_path), self.class_to_idx[c]))
            self.samples.extend(class_samples)
            self.class_counts[self.class_to_idx[c]] = len(class_samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path


# ===============================
# TRANSFORMS
# ===============================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# ===============================
# TRAINING FUNCTIONS
# ===============================

def train_model(model, train_loader, val_loader, hyperparams, device, verbose=False):
    """Train model and return validation accuracy"""
    
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=hyperparams['lr'], 
        weight_decay=hyperparams['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=hyperparams['epochs']
    )
    
    best_val_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(hyperparams['epochs']):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for imgs, lbls, _ in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            outputs = model(imgs)
            loss = F.cross_entropy(outputs, lbls, label_smoothing=hyperparams['label_smoothing'])
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += lbls.size(0)
            correct += predicted.eq(lbls).sum().item()
        
        scheduler.step()
        train_acc = 100. * correct / total
        train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                loss = F.cross_entropy(outputs, lbls)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += lbls.size(0)
                val_correct += predicted.eq(lbls).sum().item()
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        if verbose:
            print(f"    Epoch {epoch+1}/{hyperparams['epochs']} | "
                  f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
    
    return best_val_acc, history


def evaluate_model(model, test_loader, device):
    """Evaluate model and return metrics"""
    model.eval()
    
    all_preds = []
    all_trues = []
    all_probs = []
    
    with torch.no_grad():
        for imgs, lbls, _ in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_trues.extend(lbls.numpy())
            all_probs.extend(probs)
    
    all_preds = np.array(all_preds)
    all_trues = np.array(all_trues)
    
    return {
        'accuracy': accuracy_score(all_trues, all_preds),
        'balanced_accuracy': balanced_accuracy_score(all_trues, all_preds),
        'macro_f1': f1_score(all_trues, all_preds, average='macro'),
        'weighted_f1': f1_score(all_trues, all_preds, average='weighted'),
        'macro_precision': precision_score(all_trues, all_preds, average='macro', zero_division=0),
        'macro_recall': recall_score(all_trues, all_preds, average='macro', zero_division=0),
        'predictions': all_preds,
        'true_labels': all_trues,
    }


# ===============================
# LOAD DATASET
# ===============================

print("\n" + "="*70)
print("LOADING DATASET")
print("="*70)

dataset_train = FishDataset(DATA_DIR, transform=train_transform, clean_names=True)
dataset_val = FishDataset(DATA_DIR, transform=val_transform, clean_names=True)

print(f"\nTotal images: {len(dataset_train)}")
print(f"Total classes: {len(dataset_train.idx_to_class)}")

print("\nClass Distribution:")
for idx in range(len(dataset_train.idx_to_class)):
    name = dataset_train.idx_to_class[idx]
    count = dataset_train.class_counts.get(idx, 0)
    print(f"  [{idx:2d}] {name:<45} n={count}")

y = np.array([lbl for _, lbl in dataset_train.samples])


# ===============================
# NESTED CROSS-VALIDATION
# ===============================

print("\n" + "="*70)
print("STARTING NESTED CROSS-VALIDATION")
print("="*70)

outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

outer_fold_results = []
all_outer_predictions = []
all_outer_true_labels = []
best_hyperparams_per_fold = []

start_time = time.time()

for outer_fold, (outer_train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y)):
    
    print(f"\n{'='*70}")
    print(f"OUTER FOLD {outer_fold + 1}/{OUTER_FOLDS}")
    print(f"{'='*70}")
    print(f"Train: {len(outer_train_idx)}, Test: {len(test_idx)}")
    
    # Inner CV
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED + outer_fold)
    hyperparam_scores = {hp['name']: [] for hp in HYPERPARAMETER_GRID}
    
    print(f"\n  --- Inner CV for Hyperparameter Selection ---")
    
    for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(inner_cv.split(
        np.zeros(len(outer_train_idx)), y[outer_train_idx]
    )):
        print(f"\n  Inner Fold {inner_fold + 1}/{INNER_FOLDS}")
        
        actual_train_idx = outer_train_idx[inner_train_idx]
        actual_val_idx = outer_train_idx[inner_val_idx]
        
        train_subset = Subset(dataset_train, actual_train_idx)
        val_subset = Subset(dataset_val, actual_val_idx)
        
        # Class balancing
        train_labels = y[actual_train_idx]
        class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
        weights = 1.0 / (class_counts + 1e-6)
        sample_weights = weights[train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        
        for hp in HYPERPARAMETER_GRID:
            print(f"    Testing {hp['name']}...", end=" ", flush=True)
            
            model = MobileNetV3Classifier(
                num_classes=len(dataset_train.class_to_idx),
                dropout=hp['dropout'],
                pretrained=True
            ).to(device)
            
            val_acc, _ = train_model(model, train_loader, val_loader, hp, device, verbose=False)
            hyperparam_scores[hp['name']].append(val_acc)
            
            print(f"Val Acc: {val_acc:.2f}%")
            
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Select best hyperparameters
    mean_scores = {name: np.mean(scores) for name, scores in hyperparam_scores.items()}
    std_scores = {name: np.std(scores) for name, scores in hyperparam_scores.items()}
    
    best_hp_name = max(mean_scores, key=mean_scores.get)
    best_hp = next(hp for hp in HYPERPARAMETER_GRID if hp['name'] == best_hp_name)
    
    print(f"\n  --- Inner CV Results ---")
    for name in hyperparam_scores:
        print(f"    {name}: {mean_scores[name]:.2f}% ± {std_scores[name]:.2f}%")
    print(f"  ✓ Best: {best_hp_name}")
    
    best_hyperparams_per_fold.append({
        'outer_fold': outer_fold + 1,
        'best_config': best_hp_name,
        'mean_inner_score': mean_scores[best_hp_name],
        'std_inner_score': std_scores[best_hp_name],
        'all_scores': mean_scores
    })
    
    # Train final model
    print(f"\n  --- Training Final Model with {best_hp_name} ---")
    
    train_subset = Subset(dataset_train, outer_train_idx)
    test_subset = Subset(dataset_val, test_idx)
    
    train_labels = y[outer_train_idx]
    class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    final_model = MobileNetV3Classifier(
        num_classes=len(dataset_train.class_to_idx),
        dropout=best_hp['dropout'],
        pretrained=True
    ).to(device)
    
    val_acc, history = train_model(final_model, train_loader, test_loader, best_hp, device, verbose=True)
    
    # Evaluate
    print(f"\n  --- Test Results ---")
    test_metrics = evaluate_model(final_model, test_loader, device)
    
    print(f"  Accuracy:          {test_metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:          {test_metrics['macro_f1']:.4f}")
    print(f"  Weighted F1:       {test_metrics['weighted_f1']:.4f}")
    print(f"  Precision:         {test_metrics['macro_precision']:.4f}")
    print(f"  Recall:            {test_metrics['macro_recall']:.4f}")
    
    outer_fold_results.append({
        'outer_fold': outer_fold + 1,
        'best_hyperparams': best_hp_name,
        **{k: v for k, v in test_metrics.items() if k not in ['predictions', 'true_labels']},
        'history': history
    })
    
    all_outer_predictions.extend(test_metrics['predictions'])
    all_outer_true_labels.extend(test_metrics['true_labels'])
    
    del final_model

total_time = time.time() - start_time


# ===============================
# RESULTS SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION RESULTS")
print("="*70)

metrics_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 
                 'macro_precision', 'macro_recall']

print(f"\nModel: MobileNetV3-Small (Pretrained)")
print(f"Parameters: {n_params:,}")
print(f"Total time: {timedelta(seconds=int(total_time))}")

print("\n" + "-"*70)
print("Per-Fold Results:")
print("-"*70)
print(f"{'Fold':<6} {'Config':<12} {'Accuracy':>10} {'Bal Acc':>10} {'Macro F1':>10} {'W-F1':>10}")
print("-"*70)

for result in outer_fold_results:
    print(f"{result['outer_fold']:<6} {result['best_hyperparams']:<12} "
          f"{result['accuracy']:>10.4f} {result['balanced_accuracy']:>10.4f} "
          f"{result['macro_f1']:>10.4f} {result['weighted_f1']:>10.4f}")

print("\n" + "-"*70)
print("Aggregated Results (Mean ± Std):")
print("-"*70)

aggregated_results = {}
for metric in metrics_names:
    values = [r[metric] for r in outer_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci_95 = 1.96 * std_val / np.sqrt(len(values))
    aggregated_results[metric] = {'mean': mean_val, 'std': std_val, 'ci_95': ci_95, 'values': values}
    print(f"{metric:<20}: {mean_val:.4f} ± {std_val:.4f}  "
          f"(95% CI: [{mean_val-ci_95:.4f}, {mean_val+ci_95:.4f}])")

print("\n" + "-"*70)
print("Hyperparameter Selection:")
print("-"*70)
for fold_hp in best_hyperparams_per_fold:
    print(f"Fold {fold_hp['outer_fold']}: {fold_hp['best_config']} "
          f"(Inner CV: {fold_hp['mean_inner_score']:.2f}% ± {fold_hp['std_inner_score']:.2f}%)")


# ===============================
# GENERATE PLOTS
# ===============================

print("\n" + "="*70)
print("GENERATING PLOTS")
print("="*70)

plt.rcParams.update({
    'figure.dpi': 300, 
    'savefig.dpi': 300, 
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

class_names = [dataset_train.idx_to_class[i] for i in range(len(dataset_train.idx_to_class))]
all_preds = np.array(all_outer_predictions)
all_trues = np.array(all_outer_true_labels)

# 1. Confusion Matrix
print("  Generating confusion matrix...")
cm = confusion_matrix(all_trues, all_preds)
cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('(A) Confusion Matrix (Counts)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

# Add text annotations for counts
thresh = cm.max() / 2
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if cm[i, j] > 0:
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                   color='white' if cm[i, j] > thresh else 'black', fontsize=5)

ax = axes[1]
im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax.set_title('(B) Confusion Matrix (Normalized)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

# Add text annotations for normalized values
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if cm_normalized[i, j] > 0.01:
            ax.text(j, i, f'{cm_normalized[i, j]:.2f}', ha='center', va='center',
                   color='white' if cm_normalized[i, j] > 0.5 else 'black', fontsize=5)

plt.suptitle(f'MobileNetV3 - Nested CV Results ({OUTER_FOLDS} Outer × {INNER_FOLDS} Inner Folds)\n'
             f'Accuracy: {aggregated_results["accuracy"]["mean"]:.2%} ± {aggregated_results["accuracy"]["std"]:.2%}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "confusion_matrix.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 2. Performance Summary
print("  Generating performance summary...")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Metrics across folds
ax = axes[0, 0]
folds = [r['outer_fold'] for r in outer_fold_results]
metrics_to_plot = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
markers = ['o', 's', '^', 'd']

for metric, color, marker in zip(metrics_to_plot, colors, markers):
    values = [r[metric] for r in outer_fold_results]
    ax.plot(folds, values, f'-{marker}', color=color, label=metric.replace('_', ' ').title(), 
            markersize=8, linewidth=2)
    ax.axhline(y=np.mean(values), color=color, linestyle='--', alpha=0.5)

ax.set_xlabel('Outer Fold', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(A) Performance Across Folds', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.set_xticks(folds)
ax.grid(True, alpha=0.3)
# Adjust y-axis based on results
min_val = min([min([r[m] for r in outer_fold_results]) for m in metrics_to_plot])
ax.set_ylim([max(0, min_val - 0.05), 1.02])

# Box plots
ax = axes[0, 1]
data_for_box = [aggregated_results[m]['values'] for m in metrics_to_plot]
labels_box = [m.replace('_', '\n').title() for m in metrics_to_plot]

bp = ax.boxplot(data_for_box, labels=labels_box, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, data in enumerate(data_for_box):
    x = np.random.normal(i+1, 0.04, size=len(data))
    ax.scatter(x, data, c='black', alpha=0.6, s=30, zorder=3)

ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(B) Score Distribution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Per-class F1
ax = axes[1, 0]
report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
class_f1 = [report.get(str(i), {}).get('f1-score', 0) for i in range(len(class_names))]
class_colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))

bars = ax.barh(range(len(class_names)), class_f1, color=class_colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('F1-Score', fontweight='bold')
ax.set_title('(C) Per-Class F1-Score', fontweight='bold')
ax.axvline(x=np.mean(class_f1), color='red', linestyle='--', linewidth=2, 
          label=f'Mean: {np.mean(class_f1):.3f}')
ax.legend(loc='lower right')
ax.invert_yaxis()
ax.set_xlim([0, 1.05])

for i, f1 in enumerate(class_f1):
    ax.text(f1 + 0.01, i, f'{f1:.2f}', va='center', fontsize=6)

# Training curves
ax = axes[1, 1]
last_history = outer_fold_results[-1]['history']
epochs = range(1, len(last_history['train_acc']) + 1)

ax.plot(epochs, last_history['train_acc'], 'b-o', label='Train Acc', markersize=6, linewidth=2)
ax.plot(epochs, last_history['val_acc'], 'r-s', label='Val Acc', markersize=6, linewidth=2)
ax.set_xlabel('Epoch', fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title(f'(D) Training Curves (Fold {OUTER_FOLDS})', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle(f'MobileNetV3-Small Performance Analysis\n'
             f'Parameters: {n_params:,} | '
             f'Accuracy: {aggregated_results["accuracy"]["mean"]:.2%} ± {aggregated_results["accuracy"]["std"]:.2%}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "performance_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 3. Hyperparameter Analysis
print("  Generating hyperparameter analysis...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hp_names = [hp['name'] for hp in HYPERPARAMETER_GRID]
inner_scores_by_hp = {name: [] for name in hp_names}

for fold_result in best_hyperparams_per_fold:
    for hp_name, score in fold_result['all_scores'].items():
        inner_scores_by_hp[hp_name].append(score)

ax = axes[0]
positions = range(len(hp_names))
bp = ax.boxplot([inner_scores_by_hp[name] for name in hp_names], 
               positions=positions, patch_artist=True, widths=0.6)

box_colors = plt.cm.Set2(np.linspace(0, 1, len(hp_names)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(hp_names, rotation=45, ha='right')
ax.set_ylabel('Inner CV Validation Accuracy (%)', fontweight='bold')
ax.set_title('(A) Hyperparameter Performance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
hp_counts = {name: 0 for name in hp_names}
for fold_hp in best_hyperparams_per_fold:
    hp_counts[fold_hp['best_config']] += 1

bars = ax.bar(hp_names, [hp_counts[name] for name in hp_names], color=box_colors, edgecolor='black')
ax.set_ylabel('Times Selected', fontweight='bold')
ax.set_xlabel('Hyperparameter Configuration', fontweight='bold')
ax.set_title('(B) Selection Frequency', fontweight='bold')
ax.set_xticklabels(hp_names, rotation=45, ha='right')

for bar, count in zip(bars, [hp_counts[name] for name in hp_names]):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
               str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "hyperparameter_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

print("  All plots saved!")


# ===============================
# SAVE DETAILED REPORT
# ===============================

print("\n" + "="*70)
print("SAVING REPORT")
print("="*70)

report_path = os.path.join(OUTPUT_DIR, "nested_cv_report.txt")
with open(report_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("NESTED CROSS-VALIDATION REPORT\n")
    f.write("Bangladesh Fish Dataset - Data in Brief Publication\n")
    f.write("="*70 + "\n\n")
    
    f.write("MODEL INFORMATION\n")
    f.write("-"*70 + "\n")
    f.write(f"Model: MobileNetV3-Small (Pretrained on ImageNet)\n")
    f.write(f"Parameters: {n_params:,}\n")
    f.write(f"Outer Folds: {OUTER_FOLDS}\n")
    f.write(f"Inner Folds: {INNER_FOLDS}\n")
    f.write(f"Total Training Time: {timedelta(seconds=int(total_time))}\n")
    f.write(f"Device: {device}\n\n")
    
    f.write("HYPERPARAMETER SEARCH SPACE\n")
    f.write("-"*70 + "\n")
    for hp in HYPERPARAMETER_GRID:
        f.write(f"  {hp['name']}: lr={hp['lr']}, wd={hp['weight_decay']}, "
                f"epochs={hp['epochs']}, dropout={hp['dropout']}, ls={hp['label_smoothing']}\n")
    f.write("\n")
    
    f.write("AGGREGATED RESULTS (Mean ± Std)\n")
    f.write("-"*70 + "\n")
    f.write(f"{'Metric':<25} {'Mean':>10} {'Std':>10} {'95% CI':>25}\n")
    f.write("-"*70 + "\n")
    for metric in metrics_names:
        r = aggregated_results[metric]
        f.write(f"{metric:<25} {r['mean']:>10.4f} {r['std']:>10.4f} "
                f"[{r['mean']-r['ci_95']:.4f}, {r['mean']+r['ci_95']:.4f}]\n")
    f.write("\n")
    
    f.write("PER-FOLD RESULTS\n")
    f.write("-"*70 + "\n")
    f.write(f"{'Fold':<6} {'Config':<12} {'Accuracy':>10} {'Bal Acc':>10} {'Macro F1':>10}\n")
    f.write("-"*70 + "\n")
    for result in outer_fold_results:
        f.write(f"{result['outer_fold']:<6} {result['best_hyperparams']:<12} "
                f"{result['accuracy']:>10.4f} {result['balanced_accuracy']:>10.4f} "
                f"{result['macro_f1']:>10.4f}\n")
    f.write("\n")
    
    f.write("HYPERPARAMETER SELECTION\n")
    f.write("-"*70 + "\n")
    for fold_hp in best_hyperparams_per_fold:
        f.write(f"Fold {fold_hp['outer_fold']}: {fold_hp['best_config']} "
                f"(Inner CV: {fold_hp['mean_inner_score']:.2f}% ± {fold_hp['std_inner_score']:.2f}%)\n")
    f.write("\n")
    
    f.write("CLASSIFICATION REPORT\n")
    f.write("-"*70 + "\n")
    f.write(classification_report(all_trues, all_preds, target_names=class_names, zero_division=0))
    f.write("\n")
    
    f.write("CLASS INFORMATION\n")
    f.write("-"*70 + "\n")
    for idx in range(len(dataset_train.idx_to_class)):
        cleaned = dataset_train.idx_to_class[idx]
        original = dataset_train.idx_to_original_class[idx]
        count = dataset_train.class_counts.get(idx, 0)
        f1 = class_f1[idx]
        f.write(f"[{idx:2d}] {cleaned:<40} n={count:4d}  F1={f1:.3f}\n")

print(f"Report saved: {report_path}")


# ===============================
# FINAL SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION COMPLETED")
print("="*70)
print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"\nGenerated Files:")
print(f"  📊 {OUTPUT_DIR}/plots/confusion_matrix.png")
print(f"  📊 {OUTPUT_DIR}/plots/performance_analysis.png")
print(f"  📊 {OUTPUT_DIR}/plots/hyperparameter_analysis.png")
print(f"  📄 {OUTPUT_DIR}/nested_cv_report.txt")

print("\n" + "-"*70)
print("MODEL DETAILS:")
print("-"*70)
print(f"  Architecture: MobileNetV3-Small")
print(f"  Parameters: {n_params:,}")
print(f"  Pretrained: ImageNet")
print(f"  Training Time: {timedelta(seconds=int(total_time))}")

print("\n" + "-"*70)
print("FINAL RESULTS (Nested CV - Unbiased Estimate):")
print("-"*70)
print(f"  Accuracy:          {aggregated_results['accuracy']['mean']:.4f} ± {aggregated_results['accuracy']['std']:.4f}")
print(f"  Balanced Accuracy: {aggregated_results['balanced_accuracy']['mean']:.4f} ± {aggregated_results['balanced_accuracy']['std']:.4f}")
print(f"  Macro F1:          {aggregated_results['macro_f1']['mean']:.4f} ± {aggregated_results['macro_f1']['std']:.4f}")
print(f"  Weighted F1:       {aggregated_results['weighted_f1']['mean']:.4f} ± {aggregated_results['weighted_f1']['std']:.4f}")
print(f"  Precision:         {aggregated_results['macro_precision']['mean']:.4f} ± {aggregated_results['macro_precision']['std']:.4f}")
print(f"  Recall:            {aggregated_results['macro_recall']['mean']:.4f} ± {aggregated_results['macro_recall']['std']:.4f}")
print("="*70)

# Summary table for paper
print("\n" + "="*70)
print("TABLE FOR PAPER (Copy-Paste Ready):")
print("="*70)
print("""
| Metric            | Value           | 95% CI              |
|-------------------|-----------------|---------------------|""")
for metric in ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']:
    r = aggregated_results[metric]
    metric_name = metric.replace('_', ' ').title()
    print(f"| {metric_name:<17} | {r['mean']:.4f} ± {r['std']:.4f} | [{r['mean']-r['ci_95']:.4f}, {r['mean']+r['ci_95']:.4f}] |")
print("="*70)

In [ ]:
#Validating the validation


In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio
!{sys.executable} -m pip install scikit-learn matplotlib pillow tqdm umap-learn opencv-python


In [ ]:
import os, random, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ----------------------------
# CONFIG
# ----------------------------
DATA_DIR = "/path/to/Fish Dataset Raw"
OUT_DIR  = "./fish_outputs_grouped"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
FOLDS = 5
IMG_SIZE = 224
BATCH_SIZE = 24
EPOCHS = 6
LR = 3e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

IMG_EXT = {".jpg",".jpeg",".png",".bmp",".webp"}

# ----------------------------
# DATASET INDEXING (paths + labels)
# ----------------------------
def index_imagefolder(root):
    root = Path(root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    class_to_idx = {c:i for i,c in enumerate(classes)}

    items = []
    for c in classes:
        for p in (root/c).rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                items.append((str(p), class_to_idx[c]))
    return items, classes, class_to_idx

items, class_names, class_to_idx = index_imagefolder(DATA_DIR)
print("Total images:", len(items))
print("Total classes:", len(class_names))

paths = np.array([p for p,_ in items])
labels = np.array([y for _,y in items])

# ----------------------------
# A) PERCEPTUAL HASH (dHash) for near-duplicate grouping
# ----------------------------
def dhash_pil(img: Image.Image, hash_size=8):
    # dHash: resize to (hash_size+1, hash_size), compare adjacent pixels horizontally
    img = img.convert("L").resize((hash_size+1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.array(img, dtype=np.int32)
    diff = pixels[:, 1:] > pixels[:, :-1]
    # pack bits into int
    bit_string = diff.flatten().astype(np.uint8)
    h = 0
    for b in bit_string:
        h = (h << 1) | int(b)
    return h

def hamming(a:int, b:int):
    return (a ^ b).bit_count()

print("\n[1/4] Computing perceptual hashes (dHash)...")
hashes = np.zeros(len(paths), dtype=np.uint64)

for i,p in enumerate(tqdm(paths)):
    try:
        img = Image.open(p).convert("RGB")
        hashes[i] = dhash_pil(img, hash_size=8)
    except Exception:
        hashes[i] = 0

# Cluster into groups using exact hash match (fast, safe)
# (If you want approximate matching (hamming <= 3), tell me; it’s heavier.)
hash_to_gid = {}
groups = np.zeros(len(paths), dtype=np.int64)
gid = 0
for i,h in enumerate(hashes):
    if int(h) not in hash_to_gid:
        hash_to_gid[int(h)] = gid
        gid += 1
    groups[i] = hash_to_gid[int(h)]

num_groups = gid
print(f"Unique hash groups: {num_groups}")
# Show how many groups have duplicates
group_sizes = Counter(groups.tolist())
dup_groups = sum(1 for g,s in group_sizes.items() if s > 1)
max_group = max(group_sizes.values())
print(f"Groups with >=2 images (exact duplicates/near-duplicates): {dup_groups}")
print(f"Largest group size: {max_group}")

# Save a quick report
dup_report = os.path.join(OUT_DIR, "duplicate_groups_report.txt")
with open(dup_report, "w") as f:
    f.write("group_id\tcount\n")
    for g,s in group_sizes.most_common(200):
        if s > 1:
            f.write(f"{g}\t{s}\n")
print("Saved:", dup_report)

# ----------------------------
# Torch Dataset
# ----------------------------
class FishDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        y = self.labels[idx]
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y, p

# ----------------------------
# Transforms
# ----------------------------
train_tf_aug = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_tf_noaug = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ----------------------------
# Helper: class-balanced sampler
# ----------------------------
def make_weighted_sampler(y_train):
    counts = np.bincount(y_train, minlength=len(class_names))
    w = 1.0 / (counts + 1e-9)
    sample_w = w[y_train]
    return WeightedRandomSampler(torch.tensor(sample_w, dtype=torch.double),
                                 num_samples=len(y_train),
                                 replacement=True)

# ----------------------------
# Model
# ----------------------------
def build_model(num_classes):
    m = resnet50(weights=ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    yt, yp = [], []
    for x,y,_ in loader:
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(1).cpu().numpy()
        yt.append(np.array(y))
        yp.append(pred)
    yt = np.concatenate(yt)
    yp = np.concatenate(yp)
    return accuracy_score(yt, yp), f1_score(yt, yp, average="macro"), yt, yp

def train_one_epoch(model, loader, opt):
    model.train()
    total = 0.0
    for x,y,_ in loader:
        x = x.to(device)
        y = torch.tensor(y, device=device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += float(loss.item()) * x.size(0)
    return total / len(loader.dataset)

# ----------------------------
# B) GROUPED CROSS-VALIDATION
# ----------------------------
print("\n[2/4] Grouped Cross-Validation (GroupKFold) to prevent duplicate leakage...")

gkf = GroupKFold(n_splits=FOLDS)

def run_grouped_cv(use_augmentation: bool):
    fold_metrics = []
    all_true, all_pred = [], []

    for fold,(tr_idx, va_idx) in enumerate(gkf.split(paths, labels, groups), start=1):
        train_tf = train_tf_aug if use_augmentation else train_tf_noaug

        ds_train = FishDataset(paths[tr_idx], labels[tr_idx], transform=train_tf)
        ds_val   = FishDataset(paths[va_idx], labels[va_idx], transform=val_tf)

        sampler = make_weighted_sampler(labels[tr_idx])

        # ✅ Jupyter-safe: num_workers=0
        dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
        dl_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = build_model(len(class_names)).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

        best_f1, best_state = -1.0, None
        for ep in range(1, EPOCHS+1):
            loss = train_one_epoch(model, dl_train, opt)
            acc, mf1, _, _ = evaluate(model, dl_val)
            print(f"[{'AUG' if use_augmentation else 'NOAUG'}] Fold {fold}/{FOLDS} Ep {ep:02d} "
                  f"loss={loss:.4f} val_acc={acc:.4f} val_macroF1={mf1:.4f}")
            if mf1 > best_f1:
                best_f1 = mf1
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

        model.load_state_dict(best_state)
        acc, mf1, yt, yp = evaluate(model, dl_val)
        fold_metrics.append((acc, mf1))
        all_true.append(yt); all_pred.append(yp)

    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)

    mean_acc = float(np.mean([m[0] for m in fold_metrics]))
    std_acc  = float(np.std([m[0] for m in fold_metrics]))
    mean_f1  = float(np.mean([m[1] for m in fold_metrics]))
    std_f1   = float(np.std([m[1] for m in fold_metrics]))

    return (mean_acc, std_acc, mean_f1, std_f1, all_true, all_pred)


# Run BOTH: no augmentation vs augmentation (strong evidence for the paper)
noaug_res = run_grouped_cv(use_augmentation=False)
aug_res   = run_grouped_cv(use_augmentation=True)

def write_summary(tag, res):
    mean_acc, std_acc, mean_f1, std_f1, yt, yp = res
    p = os.path.join(OUT_DIR, f"grouped_cv_summary_{tag}.txt")
    with open(p, "w") as f:
        f.write(f"Grouped CV ({tag})\n")
        f.write(f"Mean Acc: {mean_acc:.4f} ± {std_acc:.4f}\n")
        f.write(f"Mean MacroF1: {mean_f1:.4f} ± {std_f1:.4f}\n\n")
        f.write(classification_report(yt, yp, target_names=class_names, digits=4))
    print("Saved:", p)

print("\nGrouped CV Summary (NO AUG):", noaug_res[:4])
print("Grouped CV Summary (AUG):   ", aug_res[:4])

write_summary("noaug", noaug_res)
write_summary("aug",   aug_res)

# Confusion matrices
def save_confmat(tag, yt, yp):
    cm = confusion_matrix(yt, yp)
    cmn = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True)+1e-9)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,10))
    plt.imshow(cmn, interpolation="nearest")
    plt.title(f"Confusion Matrix (Normalized) - {tag}")
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=90, fontsize=7)
    plt.yticks(ticks, class_names, fontsize=7)
    plt.tight_layout()
    outp = os.path.join(OUT_DIR, f"confusion_matrix_{tag}.png")
    plt.savefig(outp, dpi=220)
    plt.close()
    print("Saved:", outp)

save_confmat("noaug", noaug_res[4], noaug_res[5])
save_confmat("aug",   aug_res[4],   aug_res[5])

# ----------------------------
# C) ADVERSARIAL VALIDATION (distribution shift / split detect)
# Train a classifier to distinguish Train vs Val based on embeddings.
# If it gets >> 0.5 accuracy, your split distributions differ.
# ----------------------------
print("\n[3/4] Adversarial Validation on embeddings (train vs val separability)...")

@torch.no_grad()
def extract_embeddings_for_indices(idxs):
    tf = val_tf
    ds = FishDataset(paths[idxs], labels[idxs], transform=tf)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone = backbone.to(device)
    backbone.eval()

    feats = []
    for x,_,_ in tqdm(dl, desc="Embedding"):
        x = x.to(device)
        f = backbone(x).detach().cpu().numpy()
        feats.append(f)
    return np.concatenate(feats, axis=0)

# Use one fold split as probe (fold 1)
(tr_idx, va_idx) = next(iter(gkf.split(paths, labels, groups)))
X_tr = extract_embeddings_for_indices(tr_idx)
X_va = extract_embeddings_for_indices(va_idx)
X = np.vstack([X_tr, X_va])
y_adv = np.array([0]*len(X_tr) + [1]*len(X_va))

clf = make_pipeline(StandardScaler(with_mean=True), LogisticRegression(max_iter=2000, n_jobs=None))
clf.fit(X, y_adv)
adv_pred = clf.predict(X)
adv_acc = accuracy_score(y_adv, adv_pred)

print(f"Adversarial validation accuracy (0=train,1=val): {adv_acc:.4f}")
print("Interpretation:")
print("- ~0.50 = train/val distributions are similar (good)")
print("- >0.60 = noticeable shift (review split strategy / leakage / source bias)")

# ----------------------------
# D) Optional: save top duplicate groups for manual inspection
# ----------------------------
print("\n[4/4] Saving example file lists for duplicate groups (top 30 groups with duplicates)...")
dup_groups_sorted = [g for g,s in group_sizes.most_common() if s > 1][:30]

dup_examples_path = os.path.join(OUT_DIR, "duplicate_group_examples.txt")
with open(dup_examples_path, "w") as f:
    for g in dup_groups_sorted:
        idxs = np.where(groups == g)[0]
        f.write(f"\nGROUP {g} (count={len(idxs)}):\n")
        for i in idxs[:20]:
            f.write(f"  {paths[i]} | label={class_names[labels[i]]}\n")
print("Saved:", dup_examples_path)

print("\nDONE. Outputs saved to:", OUT_DIR)


In [ ]:
## With plot

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio
!{sys.executable} -m pip install scikit-learn matplotlib pillow tqdm umap-learn opencv-python


In [ ]:
# ==========================================================
# FISH DATASET: Advanced Augmentation + Validation + PLOTS
# Mac M1 Pro + Jupyter (MPS), Duplicate-aware Grouped CV
# ==========================================================

import os, random, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import umap
import cv2

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR   = "/path/to/Fish Dataset Raw"
OUT_DIR    = "./fish_outputs_advanced"
os.makedirs(OUT_DIR, exist_ok=True)

SEED       = 42
IMG_SIZE   = 224
BATCH_SIZE = 24
EPOCHS     = 6
FOLDS      = 5
LR         = 3e-4

# Duplicate grouping strength (fast approximate)
USE_NEAR_DUP_GROUPING = False   # set True to try near-duplicate union within buckets
HAMMING_THR = 3                 # for near-duplicate union
BUCKET_BITS = 16                # bucketing for speed (higher -> fewer comparisons)

# Plots / galleries
NUM_OUTLIER_IMAGES = 24
NUM_MISCLS_IMAGES  = 24
NUM_GRADCAM_IMAGES = 12

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

IMG_EXT = {".jpg",".jpeg",".png",".bmp",".webp"}

# -------------------------
# UTIL: index imagefolder
# -------------------------
def index_imagefolder(root):
    root = Path(root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if not classes:
        raise ValueError("No class subfolders found.")
    class_to_idx = {c:i for i,c in enumerate(classes)}
    items = []
    for c in classes:
        for p in (root/c).rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                items.append((str(p), class_to_idx[c]))
    return items, classes, class_to_idx

items, class_names, class_to_idx = index_imagefolder(DATA_DIR)
paths  = np.array([p for p,_ in items])
labels = np.array([y for _,y in items], dtype=np.int64)

print("Total images:", len(paths))
print("Total classes:", len(class_names))

# -------------------------
# PLOT: Class distribution
# -------------------------
def plot_class_distribution(labels, class_names, outpath):
    counts = np.bincount(labels, minlength=len(class_names))
    plt.figure(figsize=(12,5))
    plt.bar(np.arange(len(class_names)), counts)
    plt.xticks(np.arange(len(class_names)), class_names, rotation=90, fontsize=7)
    plt.title("Class Distribution (images per class)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

plot_class_distribution(labels, class_names, os.path.join(OUT_DIR, "class_distribution.png"))
print("Saved:", os.path.join(OUT_DIR, "class_distribution.png"))

# -------------------------
# DATASET
# -------------------------
class FishDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        y = int(self.labels[idx])
        img = Image.open(p).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y, p

# -------------------------
# TRANSFORMS (online augmentation)
# -------------------------
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# -------------------------
# MODEL BUILD
# -------------------------
def build_model(num_classes):
    m = resnet50(weights=ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# -------------------------
# EMBEDDINGS for outliers/UMAP
# -------------------------
@torch.no_grad()
def extract_embeddings(paths_subset, labels_subset):
    ds = FishDataset(paths_subset, labels_subset, transform=val_tf)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone = backbone.to(device)
    backbone.eval()

    feats = []
    ys = []
    ps = []
    for x,y,p in tqdm(dl, desc="Embedding"):
        x = x.to(device)
        f = backbone(x).detach().cpu().numpy()
        feats.append(f)
        ys.append(np.array(y, dtype=np.int64))
        ps.extend(list(p))
    return np.concatenate(feats, axis=0), np.concatenate(ys, axis=0), ps

print("\n[1] Extracting embeddings for outlier detection + UMAP...")
features, labels_embed, paths_embed = extract_embeddings(paths, labels)

# -------------------------
# OUTLIER DETECTION + plot gallery
# -------------------------
print("\n[2] Outlier detection (IsolationForest)...")
iso = IsolationForest(n_estimators=400, contamination=0.02, random_state=SEED)
pred = iso.fit_predict(features)     # -1 outlier
outlier_idx = np.where(pred == -1)[0].tolist()
print("Outliers detected:", len(outlier_idx))

# Save outlier list
outlier_txt = os.path.join(OUT_DIR, "outliers.txt")
with open(outlier_txt, "w") as f:
    for i in outlier_idx:
        f.write(f"{i}\t{labels[i]}\t{class_names[labels[i]]}\t{paths[i]}\n")
print("Saved:", outlier_txt)

def make_image_grid(image_paths, outpath, title, n=24, cols=6, thumb=160):
    sel = image_paths[:min(n, len(image_paths))]
    rows = math.ceil(len(sel)/cols)
    plt.figure(figsize=(cols*2.2, rows*2.2))
    for i,p in enumerate(sel):
        ax = plt.subplot(rows, cols, i+1)
        try:
            img = Image.open(p).convert("RGB")
            img = img.resize((thumb, thumb))
            ax.imshow(img)
            ax.axis("off")
        except Exception:
            ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

outlier_paths = [paths[i] for i in outlier_idx]
make_image_grid(outlier_paths, os.path.join(OUT_DIR, "outlier_gallery.png"),
                "Outlier Gallery (IsolationForest)", n=NUM_OUTLIER_IMAGES)
print("Saved:", os.path.join(OUT_DIR, "outlier_gallery.png"))

# -------------------------
# UMAP plot
# -------------------------
print("\n[3] UMAP plot...")
reducer = umap.UMAP(n_neighbors=20, min_dist=0.1, metric="cosine", random_state=SEED)
emb2d = reducer.fit_transform(features)

plt.figure(figsize=(9,7))
plt.scatter(emb2d[:,0], emb2d[:,1], c=labels, s=6, alpha=0.8)
plt.title("UMAP of Fish Dataset (ResNet50 embeddings)")
plt.tight_layout()
umap_path = os.path.join(OUT_DIR, "umap.png")
plt.savefig(umap_path, dpi=220)
plt.close()
print("Saved:", umap_path)

# -------------------------
# DUPLICATE GROUPING (dHash)
# -------------------------
def dhash_pil(img: Image.Image, hash_size=8):
    img = img.convert("L").resize((hash_size+1, hash_size), Image.Resampling.LANCZOS)
    px = np.array(img, dtype=np.int32)
    diff = px[:, 1:] > px[:, :-1]
    bits = diff.flatten().astype(np.uint8)
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h

def hamming(a:int, b:int):
    return (a ^ b).bit_count()

print("\n[4] Computing dHash groups...")
hashes = np.zeros(len(paths), dtype=np.uint64)
for i,p in enumerate(tqdm(paths, desc="dHash")):
    try:
        img = Image.open(p).convert("RGB")
        hashes[i] = dhash_pil(img, hash_size=8)
    except Exception:
        hashes[i] = 0

# exact grouping
hash_to_gid = {}
groups = np.zeros(len(paths), dtype=np.int64)
gid = 0
for i,h in enumerate(hashes):
    hh = int(h)
    if hh not in hash_to_gid:
        hash_to_gid[hh] = gid
        gid += 1
    groups[i] = hash_to_gid[hh]

group_sizes = Counter(groups.tolist())
dup_groups = sum(1 for g,s in group_sizes.items() if s > 1)
print(f"Unique hash groups: {gid}")
print(f"Groups with >=2 images (exact duplicates/near-duplicates): {dup_groups}")
print(f"Largest group size: {max(group_sizes.values())}")

dup_report = os.path.join(OUT_DIR, "duplicate_groups_report.txt")
with open(dup_report, "w") as f:
    f.write("group_id\tcount\n")
    for g,s in group_sizes.most_common(500):
        if s > 1:
            f.write(f"{g}\t{s}\n")
print("Saved:", dup_report)

# Optional: near-duplicate union-find within buckets (faster than all-pairs)
if USE_NEAR_DUP_GROUPING:
    print("\n[4b] Near-duplicate grouping (approx, bucketed) ...")
    parent = np.arange(len(paths), dtype=np.int64)
    rank = np.zeros(len(paths), dtype=np.int8)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a,b):
        ra, rb = find(a), find(b)
        if ra == rb: return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # bucket by top BUCKET_BITS
    buckets = defaultdict(list)
    shift = 64 - BUCKET_BITS
    for i,h in enumerate(hashes):
        key = int(h) >> shift
        buckets[key].append(i)

    # compare within bucket
    for key, idxs in tqdm(buckets.items(), desc="Bucket union"):
        if len(idxs) < 2:
            continue
        # pairwise inside bucket (bucket sizes usually small)
        for a_i in range(len(idxs)):
            a = idxs[a_i]
            ha = int(hashes[a])
            for b_i in range(a_i+1, len(idxs)):
                b = idxs[b_i]
                hb = int(hashes[b])
                if hamming(ha, hb) <= HAMMING_THR:
                    union(a,b)

    # compress to group ids
    root_to_gid = {}
    new_groups = np.zeros(len(paths), dtype=np.int64)
    ngid = 0
    for i in range(len(paths)):
        r = find(i)
        if r not in root_to_gid:
            root_to_gid[r] = ngid
            ngid += 1
        new_groups[i] = root_to_gid[r]
    groups = new_groups
    group_sizes = Counter(groups.tolist())
    dup_groups = sum(1 for g,s in group_sizes.items() if s > 1)
    print(f"Near-dup groups: {ngid} | groups with >=2: {dup_groups} | max size: {max(group_sizes.values())}")

# -------------------------
# TRAIN / EVAL HELPERS
# -------------------------
def make_weighted_sampler(y_train):
    counts = np.bincount(y_train, minlength=len(class_names))
    w = 1.0 / (counts + 1e-9)
    sample_w = w[y_train]
    return WeightedRandomSampler(
        torch.tensor(sample_w, dtype=torch.double),
        num_samples=len(y_train),
        replacement=True
    )

def train_one_epoch(model, loader, opt):
    model.train()
    total = 0.0
    for x,y,_ in loader:
        x = x.to(device)
        y = torch.tensor(y, dtype=torch.long, device=device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += float(loss.item()) * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs, all_y, all_p = [], [], []
    for x,y,p in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        all_probs.append(probs)
        all_y.append(np.array(y, dtype=np.int64))
        all_p.extend(list(p))
    return np.vstack(all_probs), np.concatenate(all_y), all_p

# -------------------------
# ADVANCED VALIDATION: Grouped CV
# Collects: fold metrics, OOF probabilities, misclassifications
# -------------------------
print("\n[5] Grouped Cross-Validation (duplicate-aware)...")

gkf = GroupKFold(n_splits=FOLDS)
fold_acc, fold_f1 = [], []
oof_probs = np.zeros((len(paths), len(class_names)), dtype=np.float32)
oof_pred  = np.zeros(len(paths), dtype=np.int64)
oof_true  = labels.copy()
miscls_records = []  # (path, true, pred, confidence)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(paths, labels, groups), start=1):
    print(f"\nFold {fold}/{FOLDS}")

    ds_tr = FishDataset(paths[tr_idx], labels[tr_idx], transform=train_tf)
    ds_va = FishDataset(paths[va_idx], labels[va_idx], transform=val_tf)

    sampler = make_weighted_sampler(labels[tr_idx])
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model(len(class_names)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    best_f1 = -1.0
    best_state = None

    for ep in range(1, EPOCHS+1):
        loss = train_one_epoch(model, dl_tr, opt)
        probs_va, y_va, p_va = predict_proba(model, dl_va)
        pred_va = probs_va.argmax(axis=1)

        acc = accuracy_score(y_va, pred_va)
        mf1 = f1_score(y_va, pred_va, average="macro")
        print(f"  Ep {ep:02d} | loss={loss:.4f} | val_acc={acc:.4f} | val_macroF1={mf1:.4f}")

        if mf1 > best_f1:
            best_f1 = mf1
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    model.load_state_dict(best_state)

    probs_va, y_va, p_va = predict_proba(model, dl_va)
    pred_va = probs_va.argmax(axis=1)

    acc = accuracy_score(y_va, pred_va)
    mf1 = f1_score(y_va, pred_va, average="macro")

    fold_acc.append(acc)
    fold_f1.append(mf1)

    # store OOF probs back into global arrays
    oof_probs[va_idx] = probs_va
    oof_pred[va_idx]  = pred_va

    # store misclassifications with confidence
    conf = probs_va.max(axis=1)
    for pth, yt, yp, cf in zip(p_va, y_va, pred_va, conf):
        if yt != yp:
            miscls_records.append((pth, int(yt), int(yp), float(cf)))

print("\nGrouped CV done.")
print("Mean Acc:", float(np.mean(fold_acc)), "Std:", float(np.std(fold_acc)))
print("Mean MacroF1:", float(np.mean(fold_f1)), "Std:", float(np.std(fold_f1)))

# Save CV summary + report
summary_txt = os.path.join(OUT_DIR, "grouped_cv_summary.txt")
with open(summary_txt, "w") as f:
    f.write("Grouped Cross-Validation (duplicate-aware)\n")
    f.write(f"Folds: {FOLDS}\n")
    f.write(f"Accuracy: mean={np.mean(fold_acc):.4f}, std={np.std(fold_acc):.4f}\n")
    f.write(f"MacroF1: mean={np.mean(fold_f1):.4f}, std={np.std(fold_f1):.4f}\n\n")
    f.write(classification_report(oof_true, oof_pred, target_names=class_names, digits=4))
print("Saved:", summary_txt)

# -------------------------
# PLOT: Fold metrics
# -------------------------
plt.figure(figsize=(8,4))
plt.plot(range(1,FOLDS+1), fold_acc, marker="o", label="Accuracy")
plt.plot(range(1,FOLDS+1), fold_f1, marker="o", label="Macro-F1")
plt.xticks(range(1,FOLDS+1))
plt.ylim(0, 1.0)
plt.title("Fold-wise Performance (Grouped CV)")
plt.legend()
plt.tight_layout()
fold_plot = os.path.join(OUT_DIR, "fold_metrics.png")
plt.savefig(fold_plot, dpi=220)
plt.close()
print("Saved:", fold_plot)

# -------------------------
# Confusion matrix (normalized)
# -------------------------
cm = confusion_matrix(oof_true, oof_pred)
cmn = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-9)

plt.figure(figsize=(12,10))
plt.imshow(cmn, interpolation="nearest")
plt.title("Confusion Matrix (Normalized) - Grouped CV")
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=90, fontsize=7)
plt.yticks(ticks, class_names, fontsize=7)
plt.tight_layout()
cm_path = os.path.join(OUT_DIR, "confusion_matrix_normalized.png")
plt.savefig(cm_path, dpi=220)
plt.close()
print("Saved:", cm_path)

# -------------------------
# Per-class F1 bar plot
# -------------------------
rep = classification_report(oof_true, oof_pred, target_names=class_names, digits=4, output_dict=True)
per_f1 = np.array([rep[c]["f1-score"] for c in class_names])

plt.figure(figsize=(12,5))
plt.bar(np.arange(len(class_names)), per_f1)
plt.xticks(np.arange(len(class_names)), class_names, rotation=90, fontsize=7)
plt.ylim(0, 1.0)
plt.title("Per-Class F1-score (Grouped CV)")
plt.tight_layout()
perclass_path = os.path.join(OUT_DIR, "per_class_f1.png")
plt.savefig(perclass_path, dpi=220)
plt.close()
print("Saved:", perclass_path)

# -------------------------
# ROC / PR curves (micro + macro) using OOF probabilities
# -------------------------
print("\n[6] ROC/PR curves (micro/macro, One-vs-Rest)...")

Y_bin = label_binarize(oof_true, classes=np.arange(len(class_names)))
# ROC micro
fpr_micro, tpr_micro, _ = roc_curve(Y_bin.ravel(), oof_probs.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)

# ROC macro
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(len(class_names)):
    fpr[i], tpr[i], _ = roc_curve(Y_bin[:,i], oof_probs[:,i])
    roc_auc[i] = auc(fpr[i], tpr[i])
roc_auc_macro = float(np.mean(list(roc_auc.values())))

plt.figure(figsize=(6,5))
plt.plot(fpr_micro, tpr_micro, label=f"micro AUC={roc_auc_micro:.3f}")
plt.plot([0,1],[0,1], linestyle="--")
plt.title("ROC Curve (micro)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
roc_path = os.path.join(OUT_DIR, "roc_micro.png")
plt.savefig(roc_path, dpi=220)
plt.close()
print("Saved:", roc_path)

# PR micro
prec_micro, rec_micro, _ = precision_recall_curve(Y_bin.ravel(), oof_probs.ravel())
ap_micro = average_precision_score(Y_bin, oof_probs, average="micro")

plt.figure(figsize=(6,5))
plt.plot(rec_micro, prec_micro, label=f"micro AP={ap_micro:.3f}")
plt.title("Precision-Recall Curve (micro)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
pr_path = os.path.join(OUT_DIR, "pr_micro.png")
plt.savefig(pr_path, dpi=220)
plt.close()
print("Saved:", pr_path)

# -------------------------
# Calibration / Reliability diagram (top-label confidence)
# -------------------------
print("\n[7] Calibration plot (reliability diagram)...")
conf = oof_probs.max(axis=1)
correct = (oof_pred == oof_true).astype(np.int32)

bins = np.linspace(0,1,11)
bin_ids = np.digitize(conf, bins) - 1
bin_acc = []
bin_conf = []
for b in range(10):
    mask = bin_ids == b
    if mask.sum() == 0:
        bin_acc.append(np.nan); bin_conf.append((bins[b]+bins[b+1])/2)
    else:
        bin_acc.append(correct[mask].mean())
        bin_conf.append(conf[mask].mean())

plt.figure(figsize=(6,5))
plt.plot([0,1],[0,1], linestyle="--", label="perfect")
plt.plot(bin_conf, bin_acc, marker="o", label="model")
plt.title("Reliability Diagram (Grouped CV OOF)")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.legend()
plt.tight_layout()
cal_path = os.path.join(OUT_DIR, "calibration.png")
plt.savefig(cal_path, dpi=220)
plt.close()
print("Saved:", cal_path)

# -------------------------
# Misclassification gallery (highest confidence wrong preds)
# -------------------------
print("\n[8] Misclassification gallery...")
miscls_records.sort(key=lambda x: x[3], reverse=True)  # high confidence wrong first
mis_paths = [r[0] for r in miscls_records[:NUM_MISCLS_IMAGES]]

def make_miscls_grid(records, outpath, n=24, cols=6, thumb=160):
    sel = records[:min(n, len(records))]
    rows = math.ceil(len(sel)/cols)
    plt.figure(figsize=(cols*2.6, rows*2.6))
    for i,(pth,yt,yp,cf) in enumerate(sel):
        ax = plt.subplot(rows, cols, i+1)
        try:
            img = Image.open(pth).convert("RGB").resize((thumb, thumb))
            ax.imshow(img)
            ax.set_title(f"T:{class_names[yt]}\nP:{class_names[yp]}\n{cf:.2f}", fontsize=7)
            ax.axis("off")
        except Exception:
            ax.axis("off")
    plt.suptitle("Top Misclassifications (High Confidence Wrong)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()

mis_grid = os.path.join(OUT_DIR, "misclassification_gallery.png")
make_miscls_grid(miscls_records, mis_grid, n=NUM_MISCLS_IMAGES)
print("Saved:", mis_grid)

# -------------------------
# Grad-CAM (final model trained on full data quickly)
# -------------------------
print("\n[9] Grad-CAM examples (final quick model)...")

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hooks = []
        self._register()

    def _register(self):
        def fwd_hook(module, inp, out):
            self.activations = out.detach()
        def bwd_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.hooks.append(self.target_layer.register_forward_hook(fwd_hook))
        self.hooks.append(self.target_layer.register_full_backward_hook(bwd_hook))

    def remove(self):
        for h in self.hooks:
            h.remove()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        w = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (w * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-9)
        return cam, logits

def train_final_model():
    ds_full = FishDataset(paths, labels, transform=train_tf)
    sampler = make_weighted_sampler(labels)
    dl = DataLoader(ds_full, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)

    m = build_model(len(class_names)).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)

    for ep in range(1, max(2, EPOCHS//2) + 1):
        loss = train_one_epoch(m, dl, opt)
        print(f"  Final Ep {ep:02d} loss={loss:.4f}")
    return m

final_model = train_final_model()
final_model.eval()

# Prepare samples: mix of random + some misclassified if available
cam_indices = list(np.random.choice(len(paths), size=min(NUM_GRADCAM_IMAGES, len(paths)), replace=False))
if len(miscls_records) > 0:
    # add a few misclassified examples (by path lookup)
    mis_extra = [r[0] for r in miscls_records[:min(6, len(miscls_records))]]
    p_to_i = {p:i for i,p in enumerate(paths)}
    cam_indices.extend([p_to_i[p] for p in mis_extra if p in p_to_i])
    cam_indices = cam_indices[:NUM_GRADCAM_IMAGES]

cam_outdir = os.path.join(OUT_DIR, "gradcam_examples")
os.makedirs(cam_outdir, exist_ok=True)

target_layer = final_model.layer4[-1].conv3
cam = GradCAM(final_model, target_layer)

for k, idx in enumerate(cam_indices):
    pth = paths[idx]
    yt  = int(labels[idx])

    # model input tensor
    img_t = val_tf(Image.open(pth).convert("RGB")).unsqueeze(0).to(device)
    heat, logits = cam(img_t)
    pred = int(logits.argmax(dim=1).item())

    heat = heat.squeeze().detach().cpu().numpy()  # HxW

    # overlay on original (opencv)
    orig = cv2.imread(pth)
    if orig is None:
        continue
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, (heat.shape[1], heat.shape[0]))

    heatmap = (heat * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = (0.55 * orig + 0.45 * heatmap).astype(np.uint8)
    outp = os.path.join(cam_outdir, f"gradcam_{k}_T{class_names[yt]}_P{class_names[pred]}.png")
    cv2.imwrite(outp, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

cam.remove()
print("Saved Grad-CAM overlays to:", cam_outdir)

print("\nDONE. All advanced outputs saved in:", OUT_DIR)


In [ ]:
# ==========================================================
# FIXED + BEAUTIFUL Saliency, Grad-CAM, Activation Heatmaps
# (Upsamples CAM properly, high-quality figures)
# ==========================================================

import os, math, random, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
import cv2

warnings.filterwarnings("ignore")

# -------------------------
# SETTINGS
# -------------------------
DATA_DIR = "/path/to/Fish Dataset Raw"
OUT_DIR  = "./fish_advanced_vis_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
IMG_SIZE = 224
N_VIS = 12
DPI = 600

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

IMG_EXT = {".jpg",".jpeg",".png",".bmp",".webp"}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "savefig.dpi": DPI,
})

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()

# -------------------------
# Index dataset
# -------------------------
def index_imagefolder(root):
    root = Path(root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    class_to_idx = {c:i for i,c in enumerate(classes)}
    items = []
    for c in classes:
        for p in (root/c).rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                items.append((str(p), class_to_idx[c]))
    return items, classes, class_to_idx

items, class_names, _ = index_imagefolder(DATA_DIR)
paths  = np.array([p for p,_ in items])
labels = np.array([y for _,y in items], dtype=np.int64)

print("Total images:", len(paths))
print("Total classes:", len(class_names))

# -------------------------
# Transforms
# -------------------------
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

vis_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
])

MEAN = np.array([0.485,0.456,0.406], dtype=np.float32)
STD  = np.array([0.229,0.224,0.225], dtype=np.float32)

def denorm_tensor(x_3hw):
    xx = x_3hw.detach().cpu().numpy().transpose(1,2,0)
    xx = (xx * STD + MEAN)
    return np.clip(xx, 0, 1)

# -------------------------
# Build model (pretrained + new head)
# NOTE: better to load YOUR trained weights if you have them.
# -------------------------
model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)
model.eval()

# -------------------------
# Smooth / resize helpers for beautiful maps
# -------------------------
def normalize01(a):
    a = a.astype(np.float32)
    a -= a.min()
    a /= (a.max() + 1e-9)
    return a

def upsample_heatmap(heat, target_hw):
    """heat: HxW float, target_hw: (H,W). returns resized heat in [0,1]."""
    heat = normalize01(heat)
    H,W = target_hw
    heat_rs = cv2.resize(heat, (W,H), interpolation=cv2.INTER_CUBIC)
    heat_rs = normalize01(heat_rs)
    # optional smoothing
    heat_rs = cv2.GaussianBlur(heat_rs, (0,0), sigmaX=1.2, sigmaY=1.2)
    return normalize01(heat_rs)

def overlay_heatmap(img_rgb01, heat01, alpha=0.42, cmap=cv2.COLORMAP_TURBO):
    """img_rgb01: HxWx3 [0,1], heat01: HxW [0,1] same spatial size."""
    h = (heat01 * 255).astype(np.uint8)
    hm = cv2.applyColorMap(h, cmap)
    hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    out = (1-alpha) * img_rgb01 + alpha * hm
    return np.clip(out, 0, 1)

# ==========================================================
# (1) SALIENCY MAP (smoothed input gradient)
# ==========================================================
def saliency_map(model, x, class_idx=None):
    model.zero_grad(set_to_none=True)
    x = x.clone().detach().requires_grad_(True)
    logits = model(x)
    pred = int(logits.argmax(dim=1).item()) if class_idx is None else int(class_idx)
    score = logits[:, pred].sum()
    score.backward()

    grad = x.grad.detach().abs().squeeze(0)   # (3,H,W)
    sal = grad.max(dim=0)[0].cpu().numpy()    # (H,W)
    sal = normalize01(sal)
    sal = cv2.GaussianBlur(sal, (0,0), 1.2)
    sal = normalize01(sal)
    return sal, pred

# ==========================================================
# (2) GRAD-CAM (returns low-res, we upsample)
# ==========================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.layer = target_layer
        self.grad = None
        self.act  = None
        self.hooks = []
        self._register()
    def _register(self):
        def fwd_hook(m, inp, out): self.act = out.detach()
        def bwd_hook(m, gin, gout): self.grad = gout[0].detach()
        self.hooks.append(self.layer.register_forward_hook(fwd_hook))
        self.hooks.append(self.layer.register_full_backward_hook(bwd_hook))
    def remove(self):
        for h in self.hooks: h.remove()
    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        pred = int(logits.argmax(dim=1).item()) if class_idx is None else int(class_idx)
        score = logits[:, pred].sum()
        score.backward(retain_graph=True)
        w = self.grad.mean(dim=(2,3), keepdim=True)          # (1,C,1,1)
        cam = (w * self.act).sum(dim=1, keepdim=False)       # (1,H,W)
        cam = F.relu(cam).squeeze(0).detach().cpu().numpy()  # (H,W) e.g., 7x7
        cam = normalize01(cam)
        return cam, pred

gc = GradCAM(model, model.layer4[-1].conv3)

# ==========================================================
# (3) ACTIVATION HEATMAPS (nice top-k channels)
# ==========================================================
@torch.no_grad()
def get_activation_maps(model, x, layer):
    acts = []
    def hook(m, inp, out): acts.append(out.detach().cpu())
    h = layer.register_forward_hook(hook)
    _ = model(x)
    h.remove()
    return acts[0].squeeze(0)  # (C,H,W)

def topk_activation_maps(act_c_hw, k=16):
    C,H,W = act_c_hw.shape
    score = act_c_hw.abs().mean(dim=(1,2))
    idx = torch.topk(score, k=min(k,C)).indices.tolist()
    maps = []
    for i in idx:
        m = act_c_hw[i].numpy()
        m = normalize01(m)
        m = cv2.GaussianBlur(m, (0,0), 0.8)
        m = normalize01(m)
        maps.append(m)
    return maps

# ==========================================================
# Generate figures
# ==========================================================
vis_idx = np.random.choice(len(paths), size=min(N_VIS, len(paths)), replace=False)

for j, idx in enumerate(vis_idx):
    pth = paths[idx]
    yt  = int(labels[idx])

    img_pil = vis_tf(Image.open(pth).convert("RGB"))
    x = val_tf(img_pil).unsqueeze(0).to(device)
    base = denorm_tensor(x.squeeze(0))  # (224,224,3)

    # Saliency
    sal, pred_s = saliency_map(model, x, None)
    sal_rs = upsample_heatmap(sal, (IMG_SIZE, IMG_SIZE))
    sal_overlay = overlay_heatmap(base, sal_rs, alpha=0.40, cmap=cv2.COLORMAP_TURBO)

    # Grad-CAM
    cam_low, pred_c = gc(x, None)          # likely 7x7
    cam_rs = upsample_heatmap(cam_low, (IMG_SIZE, IMG_SIZE))
    cam_overlay = overlay_heatmap(base, cam_rs, alpha=0.42, cmap=cv2.COLORMAP_TURBO)

    # Activation maps (mid-level layer for nicer spatial detail than layer4)
    acts = get_activation_maps(model, x, model.layer3[-1].conv3)  # (C,h,w) usually 14x14
    amap = topk_activation_maps(acts, k=16)  # list of 16 maps

    # ---- Plot: 2 rows top + activation grid bottom
    fig = plt.figure(figsize=(14, 8), facecolor="white")

    ax = plt.subplot(2,3,1)
    ax.imshow(base); ax.axis("off")
    ax.set_title(f"Original\nTrue: {class_names[yt]}", pad=8)

    ax = plt.subplot(2,3,2)
    ax.imshow(sal_overlay); ax.axis("off")
    ax.set_title(f"Saliency Overlay\nPred: {class_names[pred_s]}", pad=8)

    ax = plt.subplot(2,3,3)
    ax.imshow(cam_overlay); ax.axis("off")
    ax.set_title(f"Grad-CAM Overlay\nPred: {class_names[pred_c]}", pad=8)

    # show the raw heatmaps too
    ax = plt.subplot(2,3,5)
    ax.imshow(cam_rs, cmap="magma"); ax.axis("off")
    ax.set_title("Grad-CAM Heatmap", pad=8)

    ax = plt.subplot(2,3,4)
    ax.imshow(sal_rs, cmap="magma"); ax.axis("off")
    ax.set_title("Saliency Heatmap", pad=8)

    # Activation grid (16 maps)
    grid = plt.subplot(2,3,6)
    grid.axis("off")
    # build a mini-grid inside
    gs = grid.get_subplotspec().subgridspec(4,4, wspace=0.05, hspace=0.05)
    for k in range(16):
        axk = fig.add_subplot(gs[k//4, k%4])
        axk.imshow(upsample_heatmap(amap[k], (IMG_SIZE, IMG_SIZE)), cmap="magma")
        axk.axis("off")
    grid.set_title("Activation Heatmaps (Top-16 Channels)", pad=10)

    plt.suptitle("Explainability: Saliency + Grad-CAM + Activations", fontsize=16, y=0.98)

    outp = os.path.join(OUT_DIR, f"explainability_beautiful_{j:02d}.png")
    plt.savefig(outp, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close()

gc.remove()

print("✅ Fixed and saved beautiful explainability plots to:", OUT_DIR)


In [ ]:
## gen by claude

In [ ]:
# ===============================
# NESTED CROSS-VALIDATION PIPELINE
# 5 OUTER FOLDS × 2 INNER FOLDS
# FOR DATA IN BRIEF PUBLICATION
# ===============================

import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import time
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, 
                            classification_report, precision_score, recall_score,
                            roc_auc_score, balanced_accuracy_score)
from sklearn.preprocessing import StandardScaler
import itertools

# ===============================
# SETTINGS
# ===============================

DATA_DIR = "/path/to/Fish Dataset Raw"
OUTPUT_DIR = "./fish_outputs_nested_cv"
OUTER_FOLDS = 5
INNER_FOLDS = 2
RANDOM_SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 24

# Set random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

print("="*70)
print("NESTED CROSS-VALIDATION PIPELINE")
print("Bangladesh Fish Dataset - Data in Brief Publication")
print("="*70)
print(f"Device: {device}")
print(f"Outer Folds: {OUTER_FOLDS}")
print(f"Inner Folds: {INNER_FOLDS}")
print(f"Total model trainings: {OUTER_FOLDS * INNER_FOLDS * N_HYPERPARAMS} (per hyperparameter set)")
print("="*70)

# ===============================
# HYPERPARAMETER SEARCH SPACE
# ===============================

HYPERPARAMETER_GRID = [
    {
        'name': 'Config_1',
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'epochs': 5,
        'dropout': 0.2,
        'label_smoothing': 0.1,
        'optimizer': 'adamw'
    },
    {
        'name': 'Config_2',
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'epochs': 5,
        'dropout': 0.3,
        'label_smoothing': 0.05,
        'optimizer': 'adamw'
    },
    {
        'name': 'Config_3',
        'lr': 1e-3,
        'weight_decay': 1e-3,
        'epochs': 5,
        'dropout': 0.4,
        'label_smoothing': 0.15,
        'optimizer': 'sgd'
    },
    {
        'name': 'Config_4',
        'lr': 5e-4,
        'weight_decay': 5e-5,
        'epochs': 6,
        'dropout': 0.25,
        'label_smoothing': 0.1,
        'optimizer': 'adamw'
    },
]

N_HYPERPARAMS = len(HYPERPARAMETER_GRID)
print(f"Hyperparameter configurations: {N_HYPERPARAMS}")
for i, hp in enumerate(HYPERPARAMETER_GRID):
    print(f"  {hp['name']}: lr={hp['lr']}, wd={hp['weight_decay']}, "
          f"epochs={hp['epochs']}, dropout={hp['dropout']}")

# ===============================
# HELPER FUNCTIONS
# ===============================

def remove_bangla_from_name(name):
    """Remove Bangla/Bengali characters from class names."""
    cleaned = re.sub(r'[\u0980-\u09FF]+', '', name)
    cleaned = re.sub(r'[^\x00-\x7F]+', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'_+', '_', cleaned)
    cleaned = re.sub(r'-+', '-', cleaned)
    cleaned = cleaned.strip(' _-')
    cleaned = re.sub(r'\(\s*\)', '', cleaned)
    cleaned = re.sub(r'\[\s*\]', '', cleaned)
    cleaned = re.sub(r'\([^a-zA-Z0-9]*\)', '', cleaned)
    cleaned = cleaned.strip()
    return cleaned if cleaned else f"Species_{hash(name) % 1000}"


# ===============================
# DATASET
# ===============================

class FishDataset(Dataset):
    def __init__(self, root, transform=None, clean_names=True):
        self.root = Path(root)
        self.transform = transform
        
        classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.original_classes = classes.copy()
        
        if clean_names:
            self.cleaned_classes = [remove_bangla_from_name(c) for c in classes]
        else:
            self.cleaned_classes = classes.copy()
        
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: self.cleaned_classes[i] for i in range(len(classes))}
        self.idx_to_original_class = {i: classes[i] for i in range(len(classes))}
        
        self.samples = []
        self.class_counts = {}
        
        for c in classes:
            class_samples = []
            for img_path in (self.root / c).glob("*"):
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
                    class_samples.append((str(img_path), self.class_to_idx[c]))
            self.samples.extend(class_samples)
            self.class_counts[self.class_to_idx[c]] = len(class_samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path


# ===============================
# TRANSFORMS
# ===============================

def get_train_transform(augment_strength='medium'):
    """Get training transforms with different augmentation strengths"""
    if augment_strength == 'light':
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    elif augment_strength == 'medium':
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(degrees=25),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:  # strong
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=45),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.3),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.2)),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# ===============================
# MODEL WITH CONFIGURABLE DROPOUT
# ===============================

class FishClassifier(nn.Module):
    """ResNet50 with configurable dropout"""
    def __init__(self, num_classes, dropout=0.2, pretrained=True):
        super().__init__()
        if pretrained:
            self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        else:
            self.backbone = resnet50(weights=None)
        
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)
    
    def get_features(self, x):
        return self.backbone(x)


# ===============================
# TRAINING FUNCTION
# ===============================

def train_model(model, train_loader, val_loader, hyperparams, device, verbose=False):
    """Train model with given hyperparameters and return validation metrics"""
    
    if hyperparams['optimizer'] == 'adamw':
        optimizer = torch.optim.AdamW(
            model.parameters(), 
            lr=hyperparams['lr'], 
            weight_decay=hyperparams['weight_decay']
        )
    else:  # sgd
        optimizer = torch.optim.SGD(
            model.parameters(), 
            lr=hyperparams['lr'], 
            momentum=0.9,
            weight_decay=hyperparams['weight_decay']
        )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=hyperparams['epochs']
    )
    
    best_val_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(hyperparams['epochs']):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for imgs, lbls, _ in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            outputs = model(imgs)
            loss = F.cross_entropy(
                outputs, lbls, 
                label_smoothing=hyperparams['label_smoothing']
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += lbls.size(0)
            correct += predicted.eq(lbls).sum().item()
        
        scheduler.step()
        
        train_acc = 100. * correct / total
        train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                loss = F.cross_entropy(outputs, lbls)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += lbls.size(0)
                val_correct += predicted.eq(lbls).sum().item()
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        if verbose:
            print(f"    Epoch {epoch+1}/{hyperparams['epochs']} | "
                  f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
    
    return best_val_acc, history


def evaluate_model(model, test_loader, device):
    """Evaluate model and return detailed metrics"""
    model.eval()
    
    all_preds = []
    all_trues = []
    all_probs = []
    
    with torch.no_grad():
        for imgs, lbls, _ in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_trues.extend(lbls.numpy())
            all_probs.extend(probs)
    
    all_preds = np.array(all_preds)
    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)
    
    metrics = {
        'accuracy': accuracy_score(all_trues, all_preds),
        'balanced_accuracy': balanced_accuracy_score(all_trues, all_preds),
        'macro_f1': f1_score(all_trues, all_preds, average='macro'),
        'weighted_f1': f1_score(all_trues, all_preds, average='weighted'),
        'macro_precision': precision_score(all_trues, all_preds, average='macro', zero_division=0),
        'macro_recall': recall_score(all_trues, all_preds, average='macro', zero_division=0),
        'predictions': all_preds,
        'true_labels': all_trues,
        'probabilities': all_probs
    }
    
    return metrics


# ===============================
# NESTED CROSS-VALIDATION
# ===============================

print("\n" + "="*70)
print("STARTING NESTED CROSS-VALIDATION")
print("="*70)

# Load dataset
dataset_train = FishDataset(DATA_DIR, transform=get_train_transform('medium'), clean_names=True)
dataset_val = FishDataset(DATA_DIR, transform=val_transform, clean_names=True)

print(f"\nDataset: {len(dataset_train)} images, {len(dataset_train.idx_to_class)} classes")

# Print class distribution
print("\nClass Distribution:")
for idx in range(len(dataset_train.idx_to_class)):
    name = dataset_train.idx_to_class[idx]
    count = dataset_train.class_counts.get(idx, 0)
    print(f"  [{idx:2d}] {name:<45} n={count}")

y = np.array([lbl for _, lbl in dataset_train.samples])

# Outer CV
outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Store results
outer_fold_results = []
all_outer_predictions = []
all_outer_true_labels = []
best_hyperparams_per_fold = []

start_time = time.time()

for outer_fold, (outer_train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y)):
    
    print(f"\n{'='*70}")
    print(f"OUTER FOLD {outer_fold + 1}/{OUTER_FOLDS}")
    print(f"{'='*70}")
    print(f"Outer train size: {len(outer_train_idx)}, Test size: {len(test_idx)}")
    
    # Inner CV for hyperparameter selection
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED + outer_fold)
    
    # Track best hyperparameters
    hyperparam_scores = {hp['name']: [] for hp in HYPERPARAMETER_GRID}
    
    print(f"\n  --- Inner CV for Hyperparameter Selection ---")
    
    for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(inner_cv.split(
        np.zeros(len(outer_train_idx)), y[outer_train_idx]
    )):
        print(f"\n  Inner Fold {inner_fold + 1}/{INNER_FOLDS}")
        
        # Map inner indices to original indices
        actual_train_idx = outer_train_idx[inner_train_idx]
        actual_val_idx = outer_train_idx[inner_val_idx]
        
        print(f"    Train: {len(actual_train_idx)}, Val: {len(actual_val_idx)}")
        
        # Create data subsets
        train_subset = Subset(dataset_train, actual_train_idx)
        val_subset = Subset(dataset_val, actual_val_idx)
        
        # Class balancing
        train_labels = y[actual_train_idx]
        class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
        weights = 1.0 / (class_counts + 1e-6)
        sample_weights = weights[train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        
        # Test each hyperparameter configuration
        for hp in HYPERPARAMETER_GRID:
            print(f"    Testing {hp['name']}...", end=" ")
            
            model = FishClassifier(
                num_classes=len(dataset_train.class_to_idx),
                dropout=hp['dropout']
            ).to(device)
            
            val_acc, _ = train_model(model, train_loader, val_loader, hp, device, verbose=False)
            hyperparam_scores[hp['name']].append(val_acc)
            
            print(f"Val Acc: {val_acc:.2f}%")
            
            del model
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Select best hyperparameters based on mean inner CV score
    mean_scores = {name: np.mean(scores) for name, scores in hyperparam_scores.items()}
    std_scores = {name: np.std(scores) for name, scores in hyperparam_scores.items()}
    
    best_hp_name = max(mean_scores, key=mean_scores.get)
    best_hp = next(hp for hp in HYPERPARAMETER_GRID if hp['name'] == best_hp_name)
    
    print(f"\n  --- Inner CV Results ---")
    for name in hyperparam_scores:
        print(f"    {name}: {mean_scores[name]:.2f}% ± {std_scores[name]:.2f}%")
    print(f"  Best Hyperparameters: {best_hp_name}")
    
    best_hyperparams_per_fold.append({
        'outer_fold': outer_fold + 1,
        'best_config': best_hp_name,
        'mean_inner_score': mean_scores[best_hp_name],
        'std_inner_score': std_scores[best_hp_name],
        'all_scores': mean_scores
    })
    
    # Train final model on full outer training set with best hyperparameters
    print(f"\n  --- Training Final Model for Outer Fold {outer_fold + 1} ---")
    
    train_subset = Subset(dataset_train, outer_train_idx)
    test_subset = Subset(dataset_val, test_idx)
    
    # Class balancing
    train_labels = y[outer_train_idx]
    class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Train with best hyperparameters
    final_model = FishClassifier(
        num_classes=len(dataset_train.class_to_idx),
        dropout=best_hp['dropout']
    ).to(device)
    
    val_acc, history = train_model(
        final_model, train_loader, test_loader, 
        best_hp, device, verbose=True
    )
    
    # Evaluate on test set
    print(f"\n  --- Evaluating on Test Set ---")
    test_metrics = evaluate_model(final_model, test_loader, device)
    
    print(f"  Test Results:")
    print(f"    Accuracy:          {test_metrics['accuracy']:.4f}")
    print(f"    Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
    print(f"    Macro F1:          {test_metrics['macro_f1']:.4f}")
    print(f"    Weighted F1:       {test_metrics['weighted_f1']:.4f}")
    print(f"    Precision:         {test_metrics['macro_precision']:.4f}")
    print(f"    Recall:            {test_metrics['macro_recall']:.4f}")
    
    # Store results
    outer_fold_results.append({
        'outer_fold': outer_fold + 1,
        'best_hyperparams': best_hp_name,
        **{k: v for k, v in test_metrics.items() if k not in ['predictions', 'true_labels', 'probabilities']},
        'history': history
    })
    
    all_outer_predictions.extend(test_metrics['predictions'])
    all_outer_true_labels.extend(test_metrics['true_labels'])
    
    del final_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

total_time = time.time() - start_time

# ===============================
# RESULTS SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION RESULTS SUMMARY")
print("="*70)

# Extract metrics
metrics_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 
                 'macro_precision', 'macro_recall']

print(f"\nTotal time: {timedelta(seconds=int(total_time))}")
print(f"\nOuter Folds: {OUTER_FOLDS}, Inner Folds: {INNER_FOLDS}")

print("\n" + "-"*70)
print("Per-Fold Results:")
print("-"*70)
print(f"{'Fold':<6} {'Best Config':<12} {'Accuracy':>10} {'Bal Acc':>10} {'Macro F1':>10} {'Weighted F1':>12}")
print("-"*70)

for result in outer_fold_results:
    print(f"{result['outer_fold']:<6} {result['best_hyperparams']:<12} "
          f"{result['accuracy']:>10.4f} {result['balanced_accuracy']:>10.4f} "
          f"{result['macro_f1']:>10.4f} {result['weighted_f1']:>12.4f}")

print("\n" + "-"*70)
print("Aggregated Results (Mean ± Std):")
print("-"*70)

aggregated_results = {}
for metric in metrics_names:
    values = [r[metric] for r in outer_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci_95 = 1.96 * std_val / np.sqrt(len(values))
    aggregated_results[metric] = {'mean': mean_val, 'std': std_val, 'ci_95': ci_95, 'values': values}
    print(f"{metric:<20}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: [{mean_val-ci_95:.4f}, {mean_val+ci_95:.4f}])")

print("\n" + "-"*70)
print("Hyperparameter Selection Summary:")
print("-"*70)

hp_selection_counts = {}
for fold_hp in best_hyperparams_per_fold:
    config = fold_hp['best_config']
    hp_selection_counts[config] = hp_selection_counts.get(config, 0) + 1
    print(f"Fold {fold_hp['outer_fold']}: {config} "
          f"(Inner CV: {fold_hp['mean_inner_score']:.2f}% ± {fold_hp['std_inner_score']:.2f}%)")

print(f"\nHyperparameter selection frequency:")
for config, count in sorted(hp_selection_counts.items(), key=lambda x: -x[1]):
    print(f"  {config}: {count}/{OUTER_FOLDS} folds")


# ===============================
# GENERATE PLOTS
# ===============================

print("\n" + "="*70)
print("GENERATING PLOTS")
print("="*70)

# Set plot style
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

# 1. Confusion Matrix
print("Generating confusion matrix...")
all_preds = np.array(all_outer_predictions)
all_trues = np.array(all_outer_true_labels)
class_names = [dataset_train.idx_to_class[i] for i in range(len(dataset_train.idx_to_class))]

cm = confusion_matrix(all_trues, all_preds)
cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('(A) Confusion Matrix (Counts)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

# Normalized
ax = axes[1]
im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax.set_title('(B) Confusion Matrix (Normalized)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

plt.suptitle(f'Nested CV Results ({OUTER_FOLDS} Outer × {INNER_FOLDS} Inner Folds)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "confusion_matrix.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 2. Performance Summary Plot
print("Generating performance summary...")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Metrics across folds
ax = axes[0, 0]
folds = [r['outer_fold'] for r in outer_fold_results]
metrics_to_plot = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
markers = ['o', 's', '^', 'd']

for metric, color, marker in zip(metrics_to_plot, colors, markers):
    values = [r[metric] for r in outer_fold_results]
    ax.plot(folds, values, f'-{marker}', color=color, label=metric.replace('_', ' ').title(), 
            markersize=8, linewidth=2)
    ax.axhline(y=np.mean(values), color=color, linestyle='--', alpha=0.5)

ax.set_xlabel('Outer Fold', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(A) Performance Across Outer Folds', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xticks(folds)
ax.set_ylim([0.8, 1.02])
ax.grid(True, alpha=0.3)

# Plot 2: Box plots
ax = axes[0, 1]
data_for_box = [aggregated_results[m]['values'] for m in metrics_to_plot]
labels_box = [m.replace('_', '\n').title() for m in metrics_to_plot]

bp = ax.boxplot(data_for_box, labels=labels_box, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add individual points
for i, data in enumerate(data_for_box):
    x = np.random.normal(i+1, 0.04, size=len(data))
    ax.scatter(x, data, c='black', alpha=0.6, s=30, zorder=3)

ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(B) Score Distribution (Outer Folds)', fontweight='bold')
ax.set_ylim([0.8, 1.02])
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Per-class F1 scores
ax = axes[1, 0]
report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
class_f1 = [report.get(str(i), {}).get('f1-score', 0) for i in range(len(class_names))]
class_colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))

bars = ax.barh(range(len(class_names)), class_f1, color=class_colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('F1-Score', fontweight='bold')
ax.set_title('(C) Per-Class F1-Score', fontweight='bold')
ax.axvline(x=np.mean(class_f1), color='red', linestyle='--', linewidth=2, 
          label=f'Mean: {np.mean(class_f1):.3f}')
ax.legend(loc='lower right')
ax.invert_yaxis()
ax.set_xlim([0, 1.05])

# Add value labels
for i, f1 in enumerate(class_f1):
    ax.text(f1 + 0.01, i, f'{f1:.2f}', va='center', fontsize=6)

# Plot 4: Training curves (last fold)
ax = axes[1, 1]
last_history = outer_fold_results[-1]['history']
epochs = range(1, len(last_history['train_acc']) + 1)

ax.plot(epochs, last_history['train_acc'], 'b-o', label='Train Acc', markersize=6, linewidth=2)
ax.plot(epochs, last_history['val_acc'], 'r-s', label='Val Acc', markersize=6, linewidth=2)
ax.set_xlabel('Epoch', fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title(f'(D) Training Curves (Fold {OUTER_FOLDS})', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle(f'Nested Cross-Validation Performance Analysis\n'
             f'Accuracy: {aggregated_results["accuracy"]["mean"]:.4f} ± {aggregated_results["accuracy"]["std"]:.4f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "performance_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 3. Hyperparameter Analysis
print("Generating hyperparameter analysis...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inner CV scores for each hyperparameter
ax = axes[0]
hp_names = [hp['name'] for hp in HYPERPARAMETER_GRID]
inner_scores_by_hp = {name: [] for name in hp_names}

for fold_result in best_hyperparams_per_fold:
    for hp_name, score in fold_result['all_scores'].items():
        inner_scores_by_hp[hp_name].append(score)

positions = range(len(hp_names))
bp = ax.boxplot([inner_scores_by_hp[name] for name in hp_names], 
               positions=positions, patch_artist=True, widths=0.6)

box_colors = plt.cm.Set2(np.linspace(0, 1, len(hp_names)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(hp_names, rotation=45, ha='right')
ax.set_ylabel('Inner CV Validation Accuracy (%)', fontweight='bold')
ax.set_title('(A) Hyperparameter Performance in Inner CV', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Selection frequency
ax = axes[1]
hp_counts = {name: 0 for name in hp_names}
for fold_hp in best_hyperparams_per_fold:
    hp_counts[fold_hp['best_config']] += 1

bars = ax.bar(hp_names, [hp_counts[name] for name in hp_names], color=box_colors, edgecolor='black')
ax.set_ylabel('Times Selected', fontweight='bold')
ax.set_xlabel('Hyperparameter Configuration', fontweight='bold')
ax.set_title('(B) Hyperparameter Selection Frequency', fontweight='bold')
ax.set_xticklabels(hp_names, rotation=45, ha='right')

# Add count labels
for bar, count in zip(bars, [hp_counts[name] for name in hp_names]):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
               str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "hyperparameter_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

print("All plots saved!")


# ===============================
# SAVE DETAILED REPORT
# ===============================

print("\n" + "="*70)
print("SAVING DETAILED REPORT")
print("="*70)

report_path = os.path.join(OUTPUT_DIR, "nested_cv_report.txt")
with open(report_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("NESTED CROSS-VALIDATION REPORT\n")
    f.write("Bangladesh Fish Dataset - Data in Brief Publication\n")
    f.write("="*70 + "\n\n")
    
    f.write("EXPERIMENTAL SETUP\n")
    f.write("-"*70 + "\n")
    f.write(f"Outer Folds: {OUTER_FOLDS}\n")
    f.write(f"Inner Folds: {INNER_FOLDS}\n")
    f.write(f"Total Samples: {len(dataset_train)}\n")
    f.write(f"Total Classes: {len(dataset_train.idx_to_class)}\n")
    f.write(f"Total Training Time: {timedelta(seconds=int(total_time))}\n")
    f.write(f"Device: {device}\n\n")
    
    f.write("HYPERPARAMETER SEARCH SPACE\n")
    f.write("-"*70 + "\n")
    for hp in HYPERPARAMETER_GRID:
        f.write(f"  {hp['name']}:\n")
        for k, v in hp.items():
            if k != 'name':
                f.write(f"    {k}: {v}\n")
    f.write("\n")
    
    f.write("AGGREGATED RESULTS\n")
    f.write("-"*70 + "\n")
    f.write(f"{'Metric':<25} {'Mean':>10} {'Std':>10} {'95% CI':>25}\n")
    f.write("-"*70 + "\n")
    for metric in metrics_names:
        r = aggregated_results[metric]
        f.write(f"{metric:<25} {r['mean']:>10.4f} {r['std']:>10.4f} "
                f"[{r['mean']-r['ci_95']:.4f}, {r['mean']+r['ci_95']:.4f}]\n")
    f.write("\n")
    
    f.write("PER-FOLD RESULTS\n")
    f.write("-"*70 + "\n")
    for result in outer_fold_results:
        f.write(f"Fold {result['outer_fold']}:\n")
        f.write(f"  Best Hyperparameters: {result['best_hyperparams']}\n")
        f.write(f"  Accuracy: {result['accuracy']:.4f}\n")
        f.write(f"  Balanced Accuracy: {result['balanced_accuracy']:.4f}\n")
        f.write(f"  Macro F1: {result['macro_f1']:.4f}\n")
        f.write(f"  Weighted F1: {result['weighted_f1']:.4f}\n\n")
    
    f.write("HYPERPARAMETER SELECTION SUMMARY\n")
    f.write("-"*70 + "\n")
    for config, count in sorted(hp_selection_counts.items(), key=lambda x: -x[1]):
        f.write(f"  {config}: Selected in {count}/{OUTER_FOLDS} folds\n")
    f.write("\n")
    
    f.write("CLASSIFICATION REPORT\n")
    f.write("-"*70 + "\n")
    f.write(classification_report(all_trues, all_preds, target_names=class_names, zero_division=0))
    f.write("\n")
    
    f.write("CLASS NAMES\n")
    f.write("-"*70 + "\n")
    for idx in range(len(dataset_train.idx_to_class)):
        cleaned = dataset_train.idx_to_class[idx]
        original = dataset_train.idx_to_original_class[idx]
        count = dataset_train.class_counts.get(idx, 0)
        f.write(f"[{idx:2d}] {cleaned:<45} (n={count:4d})\n")

print(f"Report saved to: {report_path}")


# ===============================
# FINAL SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION COMPLETED")
print("="*70)
print(f"\nOutput Directory: {OUTPUT_DIR}")
print("\nGenerated Files:")
print(f"  📊 {OUTPUT_DIR}/plots/confusion_matrix.png")
print(f"  📊 {OUTPUT_DIR}/plots/performance_analysis.png")
print(f"  📊 {OUTPUT_DIR}/plots/hyperparameter_analysis.png")
print(f"  📄 {OUTPUT_DIR}/nested_cv_report.txt")

print("\n" + "-"*70)
print("FINAL RESULTS (Nested CV - Unbiased Estimate):")
print("-"*70)
print(f"  Accuracy:          {aggregated_results['accuracy']['mean']:.4f} ± {aggregated_results['accuracy']['std']:.4f}")
print(f"  Balanced Accuracy: {aggregated_results['balanced_accuracy']['mean']:.4f} ± {aggregated_results['balanced_accuracy']['std']:.4f}")
print(f"  Macro F1:          {aggregated_results['macro_f1']['mean']:.4f} ± {aggregated_results['macro_f1']['std']:.4f}")
print(f"  Weighted F1:       {aggregated_results['weighted_f1']['mean']:.4f} ± {aggregated_results['weighted_f1']['std']:.4f}")
print(f"  Precision:         {aggregated_results['macro_precision']['mean']:.4f} ± {aggregated_results['macro_precision']['std']:.4f}")
print(f"  Recall:            {aggregated_results['macro_recall']['mean']:.4f} ± {aggregated_results['macro_recall']['std']:.4f}")
print("="*70)

# Note about high accuracy
print("\n" + "="*70)
print("NOTE ON HIGH ACCURACY")
print("="*70)
print("""
If accuracy remains very high even with Nested CV, this suggests:

1. DISTINCT VISUAL FEATURES: Fish species have clearly distinguishable 
   visual characteristics (colors, patterns, body shapes).

2. GOOD DATA QUALITY: Images are well-captured with consistent conditions.

3. APPROPRIATE MODEL: ResNet50 is well-suited for this classification task.

This is VALID for a Data in Brief paper! High accuracy on a well-constructed
dataset with proper validation methodology (Nested CV) is publishable.

Consider adding to your paper:
- Discussion of inter-class visual differences
- Sample images showing distinguishing features
- Comparison with other fish datasets
- Analysis of misclassified samples (if any)
""")
print("="*70)

In [ ]:
# ===============================
# NESTED CROSS-VALIDATION PIPELINE
# 5 OUTER FOLDS × 2 INNER FOLDS
# FOR DATA IN BRIEF PUBLICATION
# ===============================

import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import time
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, 
                            classification_report, precision_score, recall_score,
                            balanced_accuracy_score)
from sklearn.preprocessing import StandardScaler

# ===============================
# SETTINGS
# ===============================

DATA_DIR = "/path/to/Fish Dataset Raw"
OUTPUT_DIR = "./fish_outputs_nested_cv"
OUTER_FOLDS = 5
INNER_FOLDS = 2
RANDOM_SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 24

# Set random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# HYPERPARAMETER SEARCH SPACE
# ===============================

HYPERPARAMETER_GRID = [
    {
        'name': 'Config_1',
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'epochs': 3,  # Reduced
        'dropout': 0.2,
        'label_smoothing': 0.1,
        'optimizer': 'adamw'
    },
    {
        'name': 'Config_2',
        'lr': 3e-4,
        'weight_decay': 1e-5,
        'epochs': 3,  # Reduced
        'dropout': 0.3,
        'label_smoothing': 0.05,
        'optimizer': 'adamw'
    },
    {
        'name': 'Config_3',
        'lr': 1e-3,
        'weight_decay': 1e-3,
        'epochs': 3,  # Reduced
        'dropout': 0.4,
        'label_smoothing': 0.15,
        'optimizer': 'sgd'
    },
    {
        'name': 'Config_4',
        'lr': 5e-4,
        'weight_decay': 5e-5,
        'epochs': 4,  # Reduced
        'dropout': 0.25,
        'label_smoothing': 0.1,
        'optimizer': 'adamw'
    },
]

N_HYPERPARAMS = len(HYPERPARAMETER_GRID)

# Now print the header
print("="*70)
print("NESTED CROSS-VALIDATION PIPELINE")
print("Bangladesh Fish Dataset - Data in Brief Publication")
print("="*70)
print(f"Device: {device}")
print(f"Outer Folds: {OUTER_FOLDS}")
print(f"Inner Folds: {INNER_FOLDS}")
print(f"Hyperparameter configurations: {N_HYPERPARAMS}")
print(f"Total model trainings: {OUTER_FOLDS * INNER_FOLDS * N_HYPERPARAMS + OUTER_FOLDS}")
print("="*70)

print("\nHyperparameter Configurations:")
for i, hp in enumerate(HYPERPARAMETER_GRID):
    print(f"  {hp['name']}: lr={hp['lr']}, wd={hp['weight_decay']}, "
          f"epochs={hp['epochs']}, dropout={hp['dropout']}, opt={hp['optimizer']}")

# ===============================
# HELPER FUNCTIONS
# ===============================

def remove_bangla_from_name(name):
    """Remove Bangla/Bengali characters from class names."""
    cleaned = re.sub(r'[\u0980-\u09FF]+', '', name)
    cleaned = re.sub(r'[^\x00-\x7F]+', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'_+', '_', cleaned)
    cleaned = re.sub(r'-+', '-', cleaned)
    cleaned = cleaned.strip(' _-')
    cleaned = re.sub(r'\(\s*\)', '', cleaned)
    cleaned = re.sub(r'\[\s*\]', '', cleaned)
    cleaned = re.sub(r'\([^a-zA-Z0-9]*\)', '', cleaned)
    cleaned = cleaned.strip()
    return cleaned if cleaned else f"Species_{hash(name) % 1000}"


# ===============================
# DATASET
# ===============================

class FishDataset(Dataset):
    def __init__(self, root, transform=None, clean_names=True):
        self.root = Path(root)
        self.transform = transform
        
        classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.original_classes = classes.copy()
        
        if clean_names:
            self.cleaned_classes = [remove_bangla_from_name(c) for c in classes]
        else:
            self.cleaned_classes = classes.copy()
        
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: self.cleaned_classes[i] for i in range(len(classes))}
        self.idx_to_original_class = {i: classes[i] for i in range(len(classes))}
        
        self.samples = []
        self.class_counts = {}
        
        for c in classes:
            class_samples = []
            for img_path in (self.root / c).glob("*"):
                if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
                    class_samples.append((str(img_path), self.class_to_idx[c]))
            self.samples.extend(class_samples)
            self.class_counts[self.class_to_idx[c]] = len(class_samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path


# ===============================
# TRANSFORMS
# ===============================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# ===============================
# MODEL WITH CONFIGURABLE DROPOUT
# ===============================

class FishClassifier(nn.Module):
    """ResNet50 with configurable dropout"""
    def __init__(self, num_classes, dropout=0.2, pretrained=True):
        super().__init__()
        if pretrained:
            self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        else:
            self.backbone = resnet50(weights=None)
        
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


# ===============================
# TRAINING FUNCTION
# ===============================

def train_model(model, train_loader, val_loader, hyperparams, device, verbose=False):
    """Train model with given hyperparameters and return validation metrics"""
    
    if hyperparams['optimizer'] == 'adamw':
        optimizer = torch.optim.AdamW(
            model.parameters(), 
            lr=hyperparams['lr'], 
            weight_decay=hyperparams['weight_decay']
        )
    else:  # sgd
        optimizer = torch.optim.SGD(
            model.parameters(), 
            lr=hyperparams['lr'], 
            momentum=0.9,
            weight_decay=hyperparams['weight_decay']
        )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=hyperparams['epochs']
    )
    
    best_val_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(hyperparams['epochs']):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for imgs, lbls, _ in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            outputs = model(imgs)
            loss = F.cross_entropy(
                outputs, lbls, 
                label_smoothing=hyperparams['label_smoothing']
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += lbls.size(0)
            correct += predicted.eq(lbls).sum().item()
        
        scheduler.step()
        
        train_acc = 100. * correct / total
        train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                loss = F.cross_entropy(outputs, lbls)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += lbls.size(0)
                val_correct += predicted.eq(lbls).sum().item()
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        if verbose:
            print(f"    Epoch {epoch+1}/{hyperparams['epochs']} | "
                  f"Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
    
    return best_val_acc, history


def evaluate_model(model, test_loader, device):
    """Evaluate model and return detailed metrics"""
    model.eval()
    
    all_preds = []
    all_trues = []
    all_probs = []
    
    with torch.no_grad():
        for imgs, lbls, _ in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(1).cpu().numpy()
            
            all_preds.extend(preds)
            all_trues.extend(lbls.numpy())
            all_probs.extend(probs)
    
    all_preds = np.array(all_preds)
    all_trues = np.array(all_trues)
    all_probs = np.array(all_probs)
    
    metrics = {
        'accuracy': accuracy_score(all_trues, all_preds),
        'balanced_accuracy': balanced_accuracy_score(all_trues, all_preds),
        'macro_f1': f1_score(all_trues, all_preds, average='macro'),
        'weighted_f1': f1_score(all_trues, all_preds, average='weighted'),
        'macro_precision': precision_score(all_trues, all_preds, average='macro', zero_division=0),
        'macro_recall': recall_score(all_trues, all_preds, average='macro', zero_division=0),
        'predictions': all_preds,
        'true_labels': all_trues,
        'probabilities': all_probs
    }
    
    return metrics


# ===============================
# LOAD DATASET
# ===============================

print("\n" + "="*70)
print("LOADING DATASET")
print("="*70)

dataset_train = FishDataset(DATA_DIR, transform=train_transform, clean_names=True)
dataset_val = FishDataset(DATA_DIR, transform=val_transform, clean_names=True)

print(f"\nTotal images: {len(dataset_train)}")
print(f"Total classes: {len(dataset_train.idx_to_class)}")

print("\nClass Distribution:")
for idx in range(len(dataset_train.idx_to_class)):
    name = dataset_train.idx_to_class[idx]
    count = dataset_train.class_counts.get(idx, 0)
    print(f"  [{idx:2d}] {name:<45} n={count}")

y = np.array([lbl for _, lbl in dataset_train.samples])


# ===============================
# NESTED CROSS-VALIDATION
# ===============================

print("\n" + "="*70)
print("STARTING NESTED CROSS-VALIDATION")
print("="*70)

# Outer CV
outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Store results
outer_fold_results = []
all_outer_predictions = []
all_outer_true_labels = []
best_hyperparams_per_fold = []

start_time = time.time()

for outer_fold, (outer_train_idx, test_idx) in enumerate(outer_cv.split(np.zeros(len(y)), y)):
    
    print(f"\n{'='*70}")
    print(f"OUTER FOLD {outer_fold + 1}/{OUTER_FOLDS}")
    print(f"{'='*70}")
    print(f"Outer train size: {len(outer_train_idx)}, Test size: {len(test_idx)}")
    
    # Inner CV for hyperparameter selection
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED + outer_fold)
    
    # Track best hyperparameters
    hyperparam_scores = {hp['name']: [] for hp in HYPERPARAMETER_GRID}
    
    print(f"\n  --- Inner CV for Hyperparameter Selection ---")
    
    for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(inner_cv.split(
        np.zeros(len(outer_train_idx)), y[outer_train_idx]
    )):
        print(f"\n  Inner Fold {inner_fold + 1}/{INNER_FOLDS}")
        
        # Map inner indices to original indices
        actual_train_idx = outer_train_idx[inner_train_idx]
        actual_val_idx = outer_train_idx[inner_val_idx]
        
        print(f"    Train: {len(actual_train_idx)}, Val: {len(actual_val_idx)}")
        
        # Create data subsets
        train_subset = Subset(dataset_train, actual_train_idx)
        val_subset = Subset(dataset_val, actual_val_idx)
        
        # Class balancing
        train_labels = y[actual_train_idx]
        class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
        weights = 1.0 / (class_counts + 1e-6)
        sample_weights = weights[train_labels]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        
        # Test each hyperparameter configuration
        for hp in HYPERPARAMETER_GRID:
            print(f"    Testing {hp['name']}...", end=" ", flush=True)
            
            model = FishClassifier(
                num_classes=len(dataset_train.class_to_idx),
                dropout=hp['dropout']
            ).to(device)
            
            val_acc, _ = train_model(model, train_loader, val_loader, hp, device, verbose=False)
            hyperparam_scores[hp['name']].append(val_acc)
            
            print(f"Val Acc: {val_acc:.2f}%")
            
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Select best hyperparameters based on mean inner CV score
    mean_scores = {name: np.mean(scores) for name, scores in hyperparam_scores.items()}
    std_scores = {name: np.std(scores) for name, scores in hyperparam_scores.items()}
    
    best_hp_name = max(mean_scores, key=mean_scores.get)
    best_hp = next(hp for hp in HYPERPARAMETER_GRID if hp['name'] == best_hp_name)
    
    print(f"\n  --- Inner CV Results ---")
    for name in hyperparam_scores:
        print(f"    {name}: {mean_scores[name]:.2f}% ± {std_scores[name]:.2f}%")
    print(f"  ✓ Best Hyperparameters: {best_hp_name}")
    
    best_hyperparams_per_fold.append({
        'outer_fold': outer_fold + 1,
        'best_config': best_hp_name,
        'mean_inner_score': mean_scores[best_hp_name],
        'std_inner_score': std_scores[best_hp_name],
        'all_scores': mean_scores
    })
    
    # Train final model on full outer training set with best hyperparameters
    print(f"\n  --- Training Final Model with {best_hp_name} ---")
    
    train_subset = Subset(dataset_train, outer_train_idx)
    test_subset = Subset(dataset_val, test_idx)
    
    # Class balancing
    train_labels = y[outer_train_idx]
    class_counts = np.bincount(train_labels, minlength=len(dataset_train.class_to_idx))
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Train with best hyperparameters
    final_model = FishClassifier(
        num_classes=len(dataset_train.class_to_idx),
        dropout=best_hp['dropout']
    ).to(device)
    
    val_acc, history = train_model(
        final_model, train_loader, test_loader, 
        best_hp, device, verbose=True
    )
    
    # Evaluate on test set
    print(f"\n  --- Test Set Evaluation ---")
    test_metrics = evaluate_model(final_model, test_loader, device)
    
    print(f"  Accuracy:          {test_metrics['accuracy']:.4f}")
    print(f"  Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
    print(f"  Macro F1:          {test_metrics['macro_f1']:.4f}")
    print(f"  Weighted F1:       {test_metrics['weighted_f1']:.4f}")
    print(f"  Precision:         {test_metrics['macro_precision']:.4f}")
    print(f"  Recall:            {test_metrics['macro_recall']:.4f}")
    
    # Store results
    outer_fold_results.append({
        'outer_fold': outer_fold + 1,
        'best_hyperparams': best_hp_name,
        **{k: v for k, v in test_metrics.items() if k not in ['predictions', 'true_labels', 'probabilities']},
        'history': history
    })
    
    all_outer_predictions.extend(test_metrics['predictions'])
    all_outer_true_labels.extend(test_metrics['true_labels'])
    
    del final_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_time = time.time() - start_time


# ===============================
# RESULTS SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION RESULTS SUMMARY")
print("="*70)

metrics_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 
                 'macro_precision', 'macro_recall']

print(f"\nTotal time: {timedelta(seconds=int(total_time))}")

print("\n" + "-"*70)
print("Per-Fold Results:")
print("-"*70)
print(f"{'Fold':<6} {'Config':<12} {'Accuracy':>10} {'Bal Acc':>10} {'Macro F1':>10} {'W-F1':>10}")
print("-"*70)

for result in outer_fold_results:
    print(f"{result['outer_fold']:<6} {result['best_hyperparams']:<12} "
          f"{result['accuracy']:>10.4f} {result['balanced_accuracy']:>10.4f} "
          f"{result['macro_f1']:>10.4f} {result['weighted_f1']:>10.4f}")

print("\n" + "-"*70)
print("Aggregated Results (Mean ± Std):")
print("-"*70)

aggregated_results = {}
for metric in metrics_names:
    values = [r[metric] for r in outer_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci_95 = 1.96 * std_val / np.sqrt(len(values))
    aggregated_results[metric] = {'mean': mean_val, 'std': std_val, 'ci_95': ci_95, 'values': values}
    print(f"{metric:<20}: {mean_val:.4f} ± {std_val:.4f}  "
          f"(95% CI: [{mean_val-ci_95:.4f}, {mean_val+ci_95:.4f}])")

print("\n" + "-"*70)
print("Hyperparameter Selection Summary:")
print("-"*70)

hp_selection_counts = {}
for fold_hp in best_hyperparams_per_fold:
    config = fold_hp['best_config']
    hp_selection_counts[config] = hp_selection_counts.get(config, 0) + 1
    print(f"Fold {fold_hp['outer_fold']}: {config} "
          f"(Inner CV: {fold_hp['mean_inner_score']:.2f}% ± {fold_hp['std_inner_score']:.2f}%)")

print(f"\nSelection frequency:")
for config, count in sorted(hp_selection_counts.items(), key=lambda x: -x[1]):
    print(f"  {config}: {count}/{OUTER_FOLDS} folds")


# ===============================
# GENERATE PLOTS
# ===============================

print("\n" + "="*70)
print("GENERATING PLOTS")
print("="*70)

plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

# Get class names
class_names = [dataset_train.idx_to_class[i] for i in range(len(dataset_train.idx_to_class))]
all_preds = np.array(all_outer_predictions)
all_trues = np.array(all_outer_true_labels)

# 1. Confusion Matrix
print("  Generating confusion matrix...")
cm = confusion_matrix(all_trues, all_preds)
cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('(A) Confusion Matrix (Counts)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

ax = axes[1]
im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax.set_title('(B) Confusion Matrix (Normalized)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=6)
ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')

plt.suptitle(f'Nested CV ({OUTER_FOLDS} Outer × {INNER_FOLDS} Inner Folds)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "confusion_matrix.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 2. Performance Summary
print("  Generating performance summary...")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Metrics across folds
ax = axes[0, 0]
folds = [r['outer_fold'] for r in outer_fold_results]
metrics_to_plot = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
markers = ['o', 's', '^', 'd']

for metric, color, marker in zip(metrics_to_plot, colors, markers):
    values = [r[metric] for r in outer_fold_results]
    ax.plot(folds, values, f'-{marker}', color=color, label=metric.replace('_', ' ').title(), 
            markersize=8, linewidth=2)
    ax.axhline(y=np.mean(values), color=color, linestyle='--', alpha=0.5)

ax.set_xlabel('Outer Fold', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(A) Performance Across Outer Folds', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.set_xticks(folds)
ax.set_ylim([0.85, 1.02])
ax.grid(True, alpha=0.3)

# Plot 2: Box plots
ax = axes[0, 1]
data_for_box = [aggregated_results[m]['values'] for m in metrics_to_plot]
labels_box = [m.replace('_', '\n').title() for m in metrics_to_plot]

bp = ax.boxplot(data_for_box, labels=labels_box, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for i, data in enumerate(data_for_box):
    x = np.random.normal(i+1, 0.04, size=len(data))
    ax.scatter(x, data, c='black', alpha=0.6, s=30, zorder=3)

ax.set_ylabel('Score', fontweight='bold')
ax.set_title('(B) Score Distribution', fontweight='bold')
ax.set_ylim([0.85, 1.02])
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Per-class F1 scores
ax = axes[1, 0]
report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
class_f1 = [report.get(str(i), {}).get('f1-score', 0) for i in range(len(class_names))]
class_colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))

bars = ax.barh(range(len(class_names)), class_f1, color=class_colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('F1-Score', fontweight='bold')
ax.set_title('(C) Per-Class F1-Score', fontweight='bold')
ax.axvline(x=np.mean(class_f1), color='red', linestyle='--', linewidth=2, 
          label=f'Mean: {np.mean(class_f1):.3f}')
ax.legend(loc='lower right')
ax.invert_yaxis()
ax.set_xlim([0, 1.05])

for i, f1 in enumerate(class_f1):
    ax.text(f1 + 0.01, i, f'{f1:.2f}', va='center', fontsize=6)

# Plot 4: Training curves (last fold)
ax = axes[1, 1]
last_history = outer_fold_results[-1]['history']
epochs = range(1, len(last_history['train_acc']) + 1)

ax.plot(epochs, last_history['train_acc'], 'b-o', label='Train Acc', markersize=6, linewidth=2)
ax.plot(epochs, last_history['val_acc'], 'r-s', label='Val Acc', markersize=6, linewidth=2)
ax.set_xlabel('Epoch', fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title(f'(D) Training Curves (Fold {OUTER_FOLDS})', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle(f'Nested CV Performance Analysis\n'
             f'Accuracy: {aggregated_results["accuracy"]["mean"]:.4f} ± {aggregated_results["accuracy"]["std"]:.4f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "performance_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

# 3. Hyperparameter Analysis
print("  Generating hyperparameter analysis...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hp_names = [hp['name'] for hp in HYPERPARAMETER_GRID]
inner_scores_by_hp = {name: [] for name in hp_names}

for fold_result in best_hyperparams_per_fold:
    for hp_name, score in fold_result['all_scores'].items():
        inner_scores_by_hp[hp_name].append(score)

ax = axes[0]
positions = range(len(hp_names))
bp = ax.boxplot([inner_scores_by_hp[name] for name in hp_names], 
               positions=positions, patch_artist=True, widths=0.6)

box_colors = plt.cm.Set2(np.linspace(0, 1, len(hp_names)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(hp_names, rotation=45, ha='right')
ax.set_ylabel('Inner CV Validation Accuracy (%)', fontweight='bold')
ax.set_title('(A) Hyperparameter Performance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
hp_counts = {name: 0 for name in hp_names}
for fold_hp in best_hyperparams_per_fold:
    hp_counts[fold_hp['best_config']] += 1

bars = ax.bar(hp_names, [hp_counts[name] for name in hp_names], color=box_colors, edgecolor='black')
ax.set_ylabel('Times Selected', fontweight='bold')
ax.set_xlabel('Hyperparameter Configuration', fontweight='bold')
ax.set_title('(B) Selection Frequency', fontweight='bold')
ax.set_xticklabels(hp_names, rotation=45, ha='right')

for bar, count in zip(bars, [hp_counts[name] for name in hp_names]):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
               str(count), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "hyperparameter_analysis.png"), 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.close()

print("  All plots saved!")


# ===============================
# SAVE REPORT
# ===============================

print("\n" + "="*70)
print("SAVING REPORT")
print("="*70)

report_path = os.path.join(OUTPUT_DIR, "nested_cv_report.txt")
with open(report_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("NESTED CROSS-VALIDATION REPORT\n")
    f.write("Bangladesh Fish Dataset - Data in Brief Publication\n")
    f.write("="*70 + "\n\n")
    
    f.write("EXPERIMENTAL SETUP\n")
    f.write("-"*70 + "\n")
    f.write(f"Outer Folds: {OUTER_FOLDS}\n")
    f.write(f"Inner Folds: {INNER_FOLDS}\n")
    f.write(f"Total Samples: {len(dataset_train)}\n")
    f.write(f"Total Classes: {len(dataset_train.idx_to_class)}\n")
    f.write(f"Total Time: {timedelta(seconds=int(total_time))}\n")
    f.write(f"Device: {device}\n\n")
    
    f.write("HYPERPARAMETER SEARCH SPACE\n")
    f.write("-"*70 + "\n")
    for hp in HYPERPARAMETER_GRID:
        f.write(f"  {hp['name']}: lr={hp['lr']}, wd={hp['weight_decay']}, "
                f"epochs={hp['epochs']}, dropout={hp['dropout']}, opt={hp['optimizer']}\n")
    f.write("\n")
    
    f.write("AGGREGATED RESULTS\n")
    f.write("-"*70 + "\n")
    f.write(f"{'Metric':<25} {'Mean':>10} {'Std':>10} {'95% CI':>25}\n")
    f.write("-"*70 + "\n")
    for metric in metrics_names:
        r = aggregated_results[metric]
        f.write(f"{metric:<25} {r['mean']:>10.4f} {r['std']:>10.4f} "
                f"[{r['mean']-r['ci_95']:.4f}, {r['mean']+r['ci_95']:.4f}]\n")
    f.write("\n")
    
    f.write("PER-FOLD RESULTS\n")
    f.write("-"*70 + "\n")
    for result in outer_fold_results:
        f.write(f"Fold {result['outer_fold']}: {result['best_hyperparams']}\n")
        f.write(f"  Accuracy: {result['accuracy']:.4f}, "
                f"Balanced Acc: {result['balanced_accuracy']:.4f}, "
                f"Macro F1: {result['macro_f1']:.4f}\n")
    f.write("\n")
    
    f.write("CLASSIFICATION REPORT\n")
    f.write("-"*70 + "\n")
    f.write(classification_report(all_trues, all_preds, target_names=class_names, zero_division=0))
    f.write("\n")
    
    f.write("CLASS NAMES\n")
    f.write("-"*70 + "\n")
    for idx in range(len(dataset_train.idx_to_class)):
        cleaned = dataset_train.idx_to_class[idx]
        count = dataset_train.class_counts.get(idx, 0)
        f.write(f"[{idx:2d}] {cleaned:<45} (n={count:4d})\n")

print(f"Report saved: {report_path}")


# ===============================
# FINAL SUMMARY
# ===============================

print("\n" + "="*70)
print("NESTED CROSS-VALIDATION COMPLETED")
print("="*70)
print(f"\nOutput: {OUTPUT_DIR}")
print(f"Time: {timedelta(seconds=int(total_time))}")

print("\n" + "-"*70)
print("FINAL RESULTS (Unbiased Estimate):")
print("-"*70)
print(f"  Accuracy:          {aggregated_results['accuracy']['mean']:.4f} ± {aggregated_results['accuracy']['std']:.4f}")
print(f"  Balanced Accuracy: {aggregated_results['balanced_accuracy']['mean']:.4f} ± {aggregated_results['balanced_accuracy']['std']:.4f}")
print(f"  Macro F1:          {aggregated_results['macro_f1']['mean']:.4f} ± {aggregated_results['macro_f1']['std']:.4f}")
print(f"  Weighted F1:       {aggregated_results['weighted_f1']['mean']:.4f} ± {aggregated_results['weighted_f1']['std']:.4f}")
print(f"  Precision:         {aggregated_results['macro_precision']['mean']:.4f} ± {aggregated_results['macro_precision']['std']:.4f}")
print(f"  Recall:            {aggregated_results['macro_recall']['mean']:.4f} ± {aggregated_results['macro_recall']['std']:.4f}")
print("="*70)